# SENDAs Agreement 1 Update 2010-2022 (Prediction, step 2.3, internal validation)

<style type="text/css">
.showopt {
background-color: #004c93; color: #FFFFFF; width: 100px; height: 20px; text-align: center; vertical-align: middle !important; float: right; font-family: sans-serif; border-radius: 8px;
}
.showopt:hover {
background-color: #dfe4f2;
color: #004c93;
}
pre.plot {
background-color: white !important;
}
.tablelines table, .tablelines td, .tablelines th {
border: 1px solid black;
}
.centrado {
text-align: center;
}
.table.center {
margin-left:auto;
margin-right:auto;
}
/* https://vivekjaiskumar.medium.com/css-is-and-not-selector-17c942ec83f :is()*/
/* Applies to outputs that are not code other than R*/
pre {
overflow-x: auto !important;
}
pre code {
word-wrap: normal !important;
white-space: pre !important;
}
/*
pre:not(.sourceCode) {
white-space: nowrap !important;
}
*/
.sourceCode { /* Important gives precedence */
font-size: 10px !important;
line-height: 50% !important;
}
body{ /* Normal */
text-align: justify;
}
.superbigimage{
overflow-y:scroll;
height:350px;
white-space: nowrap;
overflow-x: auto;
width:100%;
}
.superbigimage img{
overflow-y: scroll;
overflow-x: hidden;
}
.message { color:#446C6E; font-family: monospace;font-size: 10px; line-height: 110%; font-weight: bold;}
div.blue { background-color:#e6f0ff; border-radius: 5px; padding: 5px; text-align: justify;}
div.red { background-color:#e6bab1; border-radius: 5px; padding: 5px; text-align: justify;}
.pandoc-table { /* Should add !important; but it seems no necessary */
margin-left:auto; /* To center */
margin-right:auto;
border-collapse: collapse;
table-layout: auto;
font-size: 11px;
overflow-y: auto;
max-height:450px !important;
white-space: nowrap;
overflow-x: auto;
width:450px;
}
.pandoc-table th {/* header */
text-align: center !important;
font-size: 10px;
padding: 0px;
}
.pandoc-table td {
text-align: left !important;
font-size: 9px;
padding: 0px;
}
.pandoc-table caption {
text-align: left !important;
font-size: 11px !important;
}
.center-table {
text-align: left !important;
font-size: 9px;
overflow-y:scroll;
height:450px;
overflow-x: scroll;
}
.controlly{
overflow-y:scroll;
height:350px;
overflow-x: scroll;
}
/*2025-03-07*/
.callout-warning,
.callout-error,
.callout-message {
  font-size: 0.7rem !important;  /* Adjust this value as needed */
}
.alert, .message {
    font-size: 0.7em !important;
}
.alert.alert-warning,
.alert.alert-danger,
.message {
    font-size: 0.7em !important;
}
</style>


<style>
#| label: css-format
h1 {
    color: var(--heading-color);
    font-size: 2rem;
    margin-bottom: 1vh;
}
p {
  font-size: 1.1rem;
  line-height: 1.6rem;
}
a {
  color: var(--primary-color);
  text-decoration: none;
  border-bottom: 3px solid transparent;
  font-weight: bold;
  &:hover, &:focus {
      border-bottom: 3px solid currentColor;
  }
}
section {
  margin: 0 auto;
}
.post-meta {
  font-size: 1rem;
  font-style: italic;
  display: block;
  margin-bottom: 4vh;
  color: var(--secondary-color);
}
nav {
  display: flex;
  justify-content: flex-end;
  padding: 20px 0;
}
/*slider switch css */
.theme-switch-wrapper {
  display: flex;
  align-items: center;
  
  em {
    margin-left: 10px;
    font-size: 1rem;
  }
}
.theme-switch {
  display: inline-block;
  height: 34px;
  position: relative;
  width: 60px;
}
.theme-switch input {
  display:none;
}
.slider {
  background-color: #ccc;
  bottom: 0;
  cursor: pointer;
  left: 0;
  position: absolute;
  right: 0;
  top: 0;
  transition: .4s;
}
.slider:before {
  background-color: #fff;
  bottom: 4px;
  content: "";
  height: 26px;
  left: 4px;
  position: absolute;
  transition: .4s;
  width: 26px;
}
input:checked + .slider {
  background-color: #66bb6a;
}
input:checked + .slider:before {
  transform: translateX(26px);
}
.slider.round {
  border-radius: 34px;
}
.slider.round:before {
  border-radius: 50%;
}
</style>


<style>
.scrollable-content {
  max-height: 350px;
  overflow-y: auto;
}
pre.scrollable-code {
  max-height: 350px;
  overflow-y: auto;
}
.superbigimage {
  overflow-x: scroll;
  white-space: nowrap;
}
.superbigimage img, 
.superbigimage svg {
  max-width: none;
  height: auto;
}
</style>
<br>

# Data Loading and Exploration

## Loading Packages and uniting databases

<div class="scrollable-content">


In [ ]:
#| label: setup
#| results: "hold"
#renv falls back to copying rather than symlinking, which is evidently very slow in this configuration.
renv::settings$use.cache(FALSE)
#only use explicit dependencies (in DESCRIPTION)
renv::settings$snapshot.type("implicit")
#check if rstools is installed
if(Sys.info()["sysname"]=="Windows"){
try(installr::install.Rtools(check_r_update=F))
}
check_quarto_version <- function(required = "1.7.29", comparator = c("ge","gt","le","lt","eq")) {
  comparator <- match.arg(comparator)
  current <- package_version(paste(unlist(quarto::quarto_version()), collapse = "."))
  req     <- package_version(required)
  ok <- switch(comparator,
               ge = current >= req,
               gt = current >  req,
               le = current <= req,
               lt = current <  req,
               eq = current == req)
  if (!ok) {
    stop(sprintf("Quarto version check failed: need %s %s (installed: %s).",
                 comparator, required, current), call. = FALSE)
  }
  invisible(TRUE)
}
check_quarto_version("1.7.29", "ge") 
#change repository to CL
local({
  r <- getOption("repos")
  r["CRAN"] <- "https://cran.dcc.uchile.cl/"
  options(repos=r)
})
if(!require(pacman)){install.packages("pacman");require(pacman)}
if(!require(pak)){install.packages("pak");require(pak)}
pacman::p_unlock(lib.loc = .libPaths()) #para no tener problemas reinstalando paquetes
if(Sys.info()["sysname"]=="Windows"){
if (getRversion() != "4.4.1") { stop("Requires R version 4.4.1; Actual: ", getRversion()) }
}
#check docker
check_docker_running <- function() {
  # Try running 'docker info' to check if Docker is running
  system("docker info", intern = TRUE, ignore.stderr = TRUE)
}
if(Sys.info()["sysname"]=="Windows"){
  install_docker <- function() {
    # Open the Docker Desktop download page in the browser for installation
    browseURL("https://www.docker.com/products/docker-desktop")
  }
  # Main logic
  if (inherits(try(check_docker_running(), silent = TRUE), "try-error")) {
    liftr::install_docker()
  } else {
    message("Docker is running.")
  }
}
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
#PACKAGES#######################################################################
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
unlink("*_cache", recursive=T)
pak::pak_sitrep()
# pak::sysreqs_check_installed(unique(unlist(paks)))
#pak::lockfile_create(unique(unlist(paks)),  "dependencies_duplicates24.lock", dependencies=T)
#pak::lockfile_install("dependencies_duplicates24.lock")
#https://rdrr.io/cran/pak/man/faq.html
#pak::cache_delete()
library(tidytable)
library(ggplot2)
library(readr)
library(tableone)
library(survivalmodels)
#renv::install("patchwork@1.2.0")
library(rms)
library(survidm)
library(caret)
library(survival)
library(riskRegression)
library(prodlim)
library(caret)
library(ggplot2)
library(dplyr)
library(survex)
library(pec)
library(future)
library(future.apply)
library(parallel)
library(mice)
library(riskRegression)
library(tidyr)
library(doFuture)
library(vcd)
library(survAUC)
library(knitr)
if (!requireNamespace("ipeval", quietly = TRUE)) {
  renv::install("ipeval")
}
#flexsurv
#if(!require(compareCstat)){install.pacakges("compareCstat")}
# library(shapr)#https://norskregnesentral.github.io/shapr/
# library(SemiMarkov)#https://www.degruyterbrill.com/document/doi/10.1515/ijb-2020-0083/html?lang=en
# #https://www.jclinepi.com/article/S0895-4356(24)00142-2/fulltext
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# 3. Activate polars code completion (safe to try even if it fails)
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
#try(polars_code_completion_activate())
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# 4. BPMN from GitHub (not on CRAN, so install via devtools if missing)
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
if (!requireNamespace("bpmn", quietly = TRUE)) {
  devtools::install_github("bergant/bpmn")
}
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# 5. PhantomJS Check (use webshot if PhantomJS is missing)
#_#_#_#_#_#_#_#_#_#_#_#_#_-----------------------------
# if (!webshot::is_phantomjs_installed()) {
#   webshot::install_phantomjs()
# }
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
#FUNCTIONS######################################################################
#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_#_
#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#
# NO MORE DEBUGS
options(error = NULL)        # si antes tenías options(error = recover) o browser)
options(browserNLdisabled = FALSE)
#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#
#NAs are replaced with "" in knitr kable
options(knitr.kable.NA = '')
pander::panderOptions('big.mark', ',')
pander::panderOptions('decimal.mark', '.')

#To produce line breaks in messages and warnings
knitr::knit_hooks$set(
   error = function(x, options) {
     paste('\n\n<div class="alert alert-danger" style="font-size: small !important;">',
           gsub('##', '\n', gsub('^##\ Error', '**Error**', x)),
           '</div>', sep = '\n')
   },
   warning = function(x, options) {
     paste('\n\n<div class="alert alert-warning" style="font-size: small !important;">',
           gsub('##', '\n', gsub('^##\ Warning:', '**Warning**', x)),
           '</div>', sep = '\n')
   },
   message = function(x, options) {
     paste('<div class="message" style="font-size: small !important;">',
           gsub('##', '\n', x),
           '</div>', sep = '\n')
   }
)

tnr<- "Times New Roman" 

options(scipen=2) #display numbers rather scientific number

# ── Helpers ────────────────────────────────────────────────────────────
mode_pick_int <- function(x){
  x <- x[!is.na(x)]
  if(length(x)==0) return(NA_integer_)
  tx <- sort(table(x), decreasing = TRUE)
  as.integer(names(tx)[1L])
}
subkey_to_label <- function(x){
  y <- gsub("_"," ", tolower(x))
  y <- gsub("amphetamine type stimulants","amphetamine-type stimulants", y)
  y <- gsub("tranquilizers hypnotics","tranquilizers/hypnotics", y)
  y
}

#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:
#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:#:
find_latest_file <- function(project_root, prefix) {
  # Get all files matching the pattern
  pattern <- paste0(prefix, "_\\d{8}_\\d{4}\\.csv$")
  files <- list.files(
    path = project_root,
    pattern = pattern,
    full.names = TRUE
  )
  if (length(files) == 0) {
    stop(paste("No files found matching pattern:", pattern))
  }
  # Extract timestamps from filenames (format: YYYYMMDD_HHMM)
  extract_timestamp <- function(filename) {
    matches <- regmatches(
      basename(filename),
      gregexpr("\\d{8}_\\d{4}", basename(filename))
    )
    if (length(matches[[1]]) == 0) {
      return(NA)
    }
    timestamp_str <- matches[[1]][length(matches[[1]])]
    return(timestamp_str)
  }
  timestamps <- sapply(files, extract_timestamp)
  # Remove any files where timestamp extraction failed
  valid_idx <- !is.na(timestamps)
  files <- files[valid_idx]
  timestamps <- timestamps[valid_idx]
  if (length(files) == 0) {
    stop("No valid timestamps found in filenames")
  }
  # Sort by timestamp (descending) and return the latest
  latest_idx <- order(timestamps, decreasing = TRUE)[1]
  return(files[latest_idx])
}

# Held-out (20%) internal validation

Final clean evaluation of the two primary models on the **retained 20% test split** (seed 2125). The 80% development set is used only to fit; the 20% test set is touched only for this out-of-sample evaluation. IBS, Uno's C-index, calibration and DCA use the
same estimators as `prediction225`; an `ipeval` bootstrap cross-check is added so discrimination/calibration can be checked for consistency across two independent methods.

Everything is driven by standalone scripts under `cons/_alt_scripts/` (see `VALIDATION_HOLDOUT_README.md`).
- **best_perf1** (SHAP primary): readmission `formula_shap_readmit_clean_updated` + mortality `formula_death_updated2`
- **best_perf2** (SHAP implemented): same readmission + mortality `formula_shap_death_rule2`, but as of 2026-06-13, with a second strata on physical diagnosis.

(The readmission model is identical in both, so readmission metrics are computed once.)

In [ ]:
#| label: define-project-root-and-paths
root <- here::here()
project_root <- gsub("/cons$", "", root)
data_out <- file.path(project_root, "data", "20241015_out")
figs_out <- file.path(project_root, "cons", "_figs")
out_dir  <- file.path(project_root, "cons", "_out")

In [ ]:
#| label: set-global-options
#| message: false

options(future.globals.maxSize = 10 * 1024^3)

In [ ]:
project_root <- gsub("/cons$", "", getwd())

In [ ]:
#| label: holdout-source-engines
#| message: false
# Reproducible data layer + all held-out engines. Sourcing run_validation_holdout.R
# pulls in build_holdout_datasets(), evaluate_dual_cox_holdout(), calibrate_*_holdout(),
# ipeval_holdout(), and the audited DCA engine (run_dca_full_summary / make_dca_panel_figure).
source(paste0(project_root,"/cons/_alt_scripts/val_holdout_01_rds_to_parquet.R"))   # corrected_datasets rds -> 5 parquet (idempotent)

In [ ]:
#| label: holdout-source-engines2
#| message: false

source(file.path(project_root, 
  "cons", 
  "_alt_scripts", 
  "run_validation_holdout.R")
)

In [ ]:
#| label: holdout-source-engines3
#| message: false

source(file.path(project_root, 
  "cons", 
  "_alt_scripts", 
  "evaluate_dual_cox_holdout_dualscore.R")
)

In [ ]:
#| label: holdout-build-sets
# Rebuild TRAIN (80%) / VALIDATION (20%) from the parquet layer + the seed-2125 split
# (cons/_out/comb_split_seed2125_test20_mar26.parquet). The rebuilt train is verified
# byte-identical to the authoritative py_corrected_datasets (last row of the table).
hd <- build_holdout_datasets(force = FALSE, verify = TRUE, verbose = TRUE)
train_list <- hd$train
val_list   <- hd$val
knitr::kable(hd$checks, "markdown",
  caption = "Held-out build consistency checks (train rebuilt from parquet + split)")


### Frozen model registry for the held-out 20% (`best_perf1` / `best_perf2`)

Both configurations share the SAME readmission Cox model (`f_readmit`), fit once and
evaluated once; `best_perf1` and `best_perf2` differ only in which mortality model is
paired with it.

- **`best_perf1`**: readmission (SHAP-selected, primary) + `formula_death_updated2`
  (Full PH, primary mortality model). Expected: 26 covariate terms / 10 baseline hazard
  functions for readmission (5 treatment modalities x 2 discharge-rule-violation
  categories); 63 covariate terms / 10 baseline hazard functions for mortality. This is
  the primary configuration for the manuscript.
- **`best_perf2`**: the same readmission model + `formula_shap_death_rule2` (SHAP,
  14-variable implementation model for mortality). Expected: same readmission counts as
  above; 14 covariate terms / 10 baseline hazard functions for mortality. This is the
  parsimonious implementation alternative for mortality, not a second competing
  readmission model.

Why these two and not a third readmission configuration: the readmission decision was
already closed in development (see `seleccion_modelos_desempeno_tesis.md`); this
held-out set validates it, it does not reopen the choice. See the printed formulas
above (verbatim, from `models$best_perf1$readmit/death` and
`models$best_perf2$readmit/death`) for the exact specification, including strata terms.

A frozen, hash-verified registry (loaded from a `prediction225` closeout artifact
instead of hardcoded here) is the longer-term target but is not yet built (blocked on
`prediction225`'s own closeout step); until then, this cell's hardcoded formulas are
the authoritative source for `prediction23`.


In [ ]:
#| label: holdout-define-models

formula_death_updated2 <- Surv(death_time_from_disch_m, death_event) ~ adm_age_rec3 + porc_pobr + 
    dit_m + national_foreign + ethnicity + dg_psiq_cie_10_instudy + 
    dg_psiq_cie_10_dg + dx_f3_mood + dx_f6_personality + dx_f_any_severe_mental + 
    any_phys_dx + polysubstance_strict + sex_rec_woman + cohabitation_family_of_origin + 
    cohabitation_with_couple_children + cohabitation_others + 
    sub_dep_icd10_status_drug_dependence + any_violence_1_domestic_violence_sex_abuse + 
    tr_outcome_referral + tr_outcome_dropout + tr_outcome_adm_discharge_adm_reasons + 
    adm_motive_sanitary_sector + adm_motive_another_sud_facility_fonodrogas_senda_previene + 
    adm_motive_justice_sector + adm_motive_other + first_sub_used_alcohol + 
    first_sub_used_cocaine_paste + first_sub_used_cocaine_powder + 
    first_sub_used_other + primary_sub_mod_cocaine_paste + primary_sub_mod_cocaine_powder + 
    primary_sub_mod_alcohol + primary_sub_mod_others + tipo_de_vivienda_rec2_other_unknown + 
    occupation_condition_corr24_unemployed + occupation_condition_corr24_inactive + 
    marital_status_rec_single + marital_status_rec_separated_divorced_annulled_widowed + 
    tenure_status_household_renting + tenure_status_household_others + 
    tenure_status_household_stays_temporarily_with_a_relative + 
    tenure_status_household_illegal_settlement + urbanicity_cat_2_mixed + 
    urbanicity_cat_1_rural + evaluacindelprocesoteraputico_logro_intermedio + 
    eva_consumo_logro_intermedio + eva_consumo_logro_minimo + 
    eva_fam_logro_intermedio + eva_fam_logro_minimo + eva_relinterp_logro_intermedio + 
    eva_relinterp_logro_minimo + eva_ocupacion_logro_intermedio + 
    eva_ocupacion_logro_minimo + eva_sm_logro_intermedio + eva_sm_logro_minimo + 
    eva_fisica_logro_intermedio + eva_fisica_logro_minimo + eva_transgnorma_logro_intermedio + 
    eva_transgnorma_logro_minimo + prim_sub_freq_rec_2_2_6_days_wk + 
    prim_sub_freq_rec_3_daily + ed_attainment_corr_2_completed_high_school_or_less + 
    ed_attainment_corr_3_completed_primary_school_or_less + strata(plan_type_strata) + 
    strata(tr_outcome_adm_discharge_rule_violation_undet)

formula_shap_death_rule2 <- Surv(death_time_from_disch_m, death_event) ~ adm_age_rec3 + primary_sub_mod_cocaine_paste + 
    primary_sub_mod_cocaine_powder + primary_sub_mod_alcohol + 
    primary_sub_mod_others + prim_sub_freq_rec_2_2_6_days_wk + 
    prim_sub_freq_rec_3_daily + occupation_condition_corr24_unemployed + 
    occupation_condition_corr24_inactive + eva_ocupacion_logro_intermedio + 
    eva_ocupacion_logro_minimo + cohabitation_family_of_origin + 
    cohabitation_with_couple_children + cohabitation_others + 
    strata(plan_type_strata) + strata(any_phys_dx)

f_readmit <- Surv(readmit_time_from_disch_m, readmit_event) ~ primary_sub_mod_cocaine_paste + 
    primary_sub_mod_cocaine_powder + primary_sub_mod_alcohol + 
    primary_sub_mod_others + adm_age_rec3 + porc_pobr + sex_rec_woman + 
    ethnicity + dit_m + eva_consumo_logro_intermedio + eva_consumo_logro_minimo + 
    ed_attainment_corr_2_completed_high_school_or_less + ed_attainment_corr_3_completed_primary_school_or_less + 
    occupation_condition_corr24_unemployed + occupation_condition_corr24_inactive + 
    dg_psiq_cie_10_dg + sub_dep_icd10_status_drug_dependence + 
    polysubstance_strict + eva_sm_logro_intermedio + eva_sm_logro_minimo + 
    evaluacindelprocesoteraputico_logro_intermedio + 
    tr_outcome_referral + tr_outcome_dropout + tr_outcome_adm_discharge_adm_reasons + 
    prim_sub_freq_rec_2_2_6_days_wk + prim_sub_freq_rec_3_daily + 
    strata(plan_type_strata) + strata(tr_outcome_adm_discharge_rule_violation_undet)


models <- list(
  best_perf1 = list(readmit = f_readmit, death = formula_death_updated2),  # SHAP readmit + Full PH death
  best_perf2 = list(readmit = f_readmit, death = formula_shap_death_rule2)        # SHAP readmit + SHAP death
)
EVAL_TIMES   <- c(3, 6, 12, 36, 60)  # C-index/IBS-over-time grid, 2026-03-18
DCA_HORIZONS <- c(6, 12, 36, 60)
CAL_TIMES    <- c(6, 12, 36, 60)


## C-index and IBS (held-out 20%)

Fit on TRAIN imputation *i*, predict on VALIDATION imputation *i*, pool across the 5 imputations. Reuses `.dual_cox_fit_one_risk` (Uno's C via `survival::concordance(timewt="n/G2", reverse=TRUE)`; IBS via IPCW) from `evaluate_dual_cox_python_style_boot.R`.

In [ ]:
models$best_perf1$readmit

In [ ]:
models$best_perf1$death

In [ ]:
models$best_perf2$readmit

In [ ]:
models$best_perf2$death

In [ ]:
#| label: holdout-cindex-ibs-run

results_boot_val_bp1 <- evaluate_dual_cox_holdout_dualscore(
  models$best_perf1$readmit, models$best_perf1$death,
  train_list, val_list, eval_times = EVAL_TIMES)
results_boot_val_bp2 <- evaluate_dual_cox_holdout_dualscore(
  models$best_perf2$readmit, models$best_perf2$death,
  train_list, val_list, eval_times = EVAL_TIMES)


In [ ]:
#| label: holdout-baseline-stratum
.t0 <- Sys.time()

# Baseline table per stratum at fixed horizons (6/12/36/60 months).
# Per imputation: refit the Cox model, extract the Breslow cumulative baseline
# hazard per stratum and evaluate it at the horizons; then average across
# imputations (pragmatic MI pooling of H0 -- declare this in the text).
# n/events are the across-imputation means, rounded.

bh_table <- function(f, train_list, times = c(6, 12, 36, 60)) {
  tl <- attr(stats::terms(f), "term.labels")
  sv <- gsub("^strata\\(|\\)$", "", tl[grepl("^strata\\(", tl)])
  status_var <- all.vars(f[[2]])[2]   # status from Surv(time, status)

  per_imp <- lapply(train_list, function(d) {
        fit <- survival::coxph(f, data = d, model = TRUE)
    bh  <- survival::basehaz(fit, centered = FALSE)
    # Evaluate the step function at the horizons (last jump <= t; flat tail)
    H <- sapply(split(bh[, c("time", "hazard")], bh$strata), function(z)
      stats::approx(z$time, z$hazard, xout = times, method = "constant", f = 0, rule = 2)$y)
    # Rebuild the combined stratum label per subject (coxph convention "v1=l1, v2=l2")
    lab <- do.call(paste, c(lapply(sv, function(v) paste0(v, "=", as.character(d[[v]]))), sep = ", "))
    if (!setequal(colnames(H), unique(lab)))
      stop("Stratum labels from basehaz() do not match the data; inspect the strata variables.")
    list(H = H, n = table(lab), ev = tapply(d[[status_var]], lab, sum))
  })

  g <- sort(colnames(per_imp[[1]]$H))
  if (!all(vapply(per_imp, function(x) setequal(colnames(x$H), g), logical(1))))
    stop("Stratum set differs across imputations; inspect imputed strata variables before pooling.")

  Hm <- Reduce(`+`, lapply(per_imp, function(x) x$H[, g, drop = FALSE])) / length(per_imp)
  n_mean  <- rowMeans(vapply(per_imp, function(x) as.numeric(x$n[g]),  numeric(length(g))))
  ev_mean <- rowMeans(vapply(per_imp, function(x) as.numeric(x$ev[g]), numeric(length(g))))

  out <- data.frame(stratum = g, n = round(n_mean), events = round(ev_mean), check.names = FALSE)
  for (k in seq_along(times)) {
    out[[paste0("H0_", times[k], "m")]] <- round(Hm[k, g], 4)
    out[[paste0("S0_", times[k], "m")]] <- round(exp(-Hm[k, g]), 4)
  }
  rownames(out) <- NULL
  out
}

tab_readmit <- bh_table(models$best_perf1$readmit, train_list)
tab_death1  <- bh_table(models$best_perf1$death,   train_list)
tab_death2  <- bh_table(models$best_perf2$death,   train_list)

knitr::kable(tab_readmit, "markdown", caption = "Baseline (Breslow, MI-pooled) per stratum: readmission")
knitr::kable(tab_death1,  "markdown", caption = "Baseline (Breslow, MI-pooled) per stratum: death, best_perf1 (Full PH)")
knitr::kable(tab_death2,  "markdown", caption = "Baseline (Breslow, MI-pooled) per stratum: death, best_perf2 (SHAP)")

cat("Elapsed (min):", round(as.numeric(difftime(Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: holdout-cindex-ibs-global
cindex_ibs_global <- dplyr::bind_rows(
  dplyr::mutate(dplyr::filter(results_boot_val_bp1$summary, Time == "Global"), model = "best_perf1"),
  dplyr::mutate(dplyr::filter(results_boot_val_bp2$summary, Time == "Global"), model = "best_perf2")
) |>
  dplyr::select(model, Risk, Metric, mean, sd, q025, q975)
knitr::kable(cindex_ibs_global, "markdown", digits = 4,
  caption = "Held-out 20%: global Uno's C-index and IBS (mean/sd across imputations)")


In [ ]:
#| label: holdout-cindex-ibs-plot
# NOTE: error bars are the across-imputation spread (tiny: outcomes are shared across
# imputations). Sampling uncertainty is given by the ipeval bootstrap CIs further below.

set1_colors <- rev(RColorBrewer::brewer.pal(3, "Set1")[1:2])
plot_metrics <- function(s, ttl) {
  df <- dplyr::filter(s, Time != "Global") |>
    dplyr::mutate(
      Time_num = as.numeric(as.character(Time)),
      Risk = factor(Risk, levels = c("Readmission", "Death")),
      Metric = ifelse(Metric == "Uno's C-Index", "Discrimination (C-index)", "Prediction error (IBS)"),
      ScoreType = ifelse(is.na(ScoreType), "n/a", ScoreType),
      # nudge lp vs risk +-1 month on the x-axis so points/error bars at the same
      # horizon don't sit exactly on top of each other; IBS ("n/a", single score)
      # is not nudged.
      Time_plot = Time_num + dplyr::case_when(ScoreType == "risk" ~ -1, ScoreType == "lp" ~ 1, TRUE ~ 0)
    )

  n_imp <- max(s$n, na.rm = TRUE)
  message(
    "holdout-cindex-ibs-plot: solid line = risk (1-S(t) at each horizon, PRIMARY score under stratified baseline ",
    "hazards); dashed line = lp (linear predictor, audit/stability check). Error bars show the 2.5-97.5 percentile of ",
    "`mean` ACROSS THE ", n_imp, " IMPUTATIONS (q025/q975 columns) -- NOT a bootstrap sampling CI; with only ", n_imp,
    " imputations this spread is small by construction and must not be read as sampling uncertainty (the ipeval/",
    "bootstrap cells further below give the real sampling CIs). lp and risk are nudged +-1 month on the x-axis so ",
    "they don't overlap when close in value."
  )

 ggplot(df, aes(Time_plot, mean, color = Risk, fill = Risk, linetype = ScoreType,
                 group = interaction(Risk, ScoreType))) +
    geom_line(linewidth = 1) +
    geom_point(size = 1.5) +
    geom_errorbar(aes(ymin = q025, ymax = q975), width = 1.5, alpha = 0.6) +
    facet_wrap(~Metric, scales = "free_y") +
    scale_color_manual(values = set1_colors) + scale_fill_manual(values = set1_colors) +
    scale_linetype_manual(values = c(risk = "solid", lp = "dashed", `n/a` = "solid"), breaks = c("risk", "lp")) +
    scale_x_continuous(breaks = seq(0, 108, 12)) +
    labs(x = "Months since discharge", y = "Value (2.5-97.5 percentile across imputations)",
         color = "Outcome", fill = "Outcome", linetype = "Score", title = ttl) +
    theme_classic(base_size = 14) + theme(legend.position = "bottom")
}
plot_metrics(results_boot_val_bp1$summary, NULL)#"best_perf1 (SHAP readmit + Full PH death)")
plot_metrics(results_boot_val_bp2$summary, NULL)#"best_perf2 (SHAP readmit + SHAP death)")


The existing `#| label: holdout-cindex-bootstrap` cell ranks Uno's C on the linear predictor `lp_val`. With stratified baseline hazards, `1 - S(t)` is not a monotone transform of `lp`, so the concordance ranking is  horizon-specific and differs from the `lp` ranking. The CV pipeline reports performance under `options(dualcox.concordance_score = "risk")`, i.e. on absolute risk.

In [ ]:
#| label: holdout-cindex-delta-death-boot

suppressPackageStartupMessages(library(survival))
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/delta_c_holdout.R"))
stopifnot(exists("results_boot_val_bp1"), exists("results_boot_val_bp2"))

DELTA_C_HORIZONS <- c(6, 12, 36, 60)
B_DELTA_C        <- 1000L        # smoke-test with 100 first
DELTA_C_SEED     <- 2125L

# DEATH: best_perf1 (Full PH) - best_perf2 (SHAP). delta_C > 0 favours best_perf1.
delta_c_death <- delta_c_holdout_absrisk(
  results_boot_val_bp1, results_boot_val_bp2, outcome = "death",
  horizons = DELTA_C_HORIZONS, B = B_DELTA_C, seed = DELTA_C_SEED,
  label_A = "best_perf1 (Full PH)", label_B = "best_perf2 (SHAP)")

# READMISSION is shared between the two best_perf models -> delta C must be 0.
delta_c_readmit <- delta_c_holdout_absrisk(
  results_boot_val_bp1, results_boot_val_bp2, outcome = "readmission",
  horizons = DELTA_C_HORIZONS, B = B_DELTA_C, seed = DELTA_C_SEED,
  label_A = "best_perf1", label_B = "best_perf2", verbose = FALSE)
stopifnot(max(abs(delta_c_readmit$delta_C), na.rm = TRUE) < 1e-8)

if (exists("out_dir"))
  utils::write.csv(delta_c_death,
                   file.path(out_dir, "holdout_delta_c_absrisk_death.csv"),
                   row.names = FALSE)

In [ ]:
cat("\n== Held-out paired delta C on absolute risk 1 - S(t) | DEATH ==\n")
knitr::kable(delta_c_death[, c("horizon", "C_A", "C_B", "delta_C",
            "delta_C_lower", "delta_C_upper", "excludes_zero", "favours")],
    "markdown", caption="Held-out paired delta C on absolute risk 1 - S(t) ", digits=3) |> print()

cat("\ndelta_C = C(best_perf1) - C(best_perf2); paired patient bootstrap of the held-out",
    "test set (model frozen, no refit), MI-pooled over imputations. 95% CI = 2.5/97.5",
    "percentile of the paired difference.\n")

## Calibration (held-out 20%)

For readmission, three predicted-risk versions are compared against the same observed incidence, estimated via Aalen-Johansen within deciles with death as a competing event: the cause-specific net risk (one minus survival from the readmission Cox model), the joint cumulative incidence obtained via `riskRegression::CSC` pairing readmission with the full mortality model (`updated2`), and the same joint incidence pairing readmission with the SHAP mortality model (`rule2`). For mortality, the single pathway is kept, `coxph → predictRisk → Kaplan-Meier`, with ICI via loess (span 0.75).

In [ ]:
#| label: holdout-calibration-run
#| message: false
source(file.path(project_root, "cons/_alt_scripts/holdout_cif_cache.R"))

cif_cache_bp1 <- .holdout_build_cif_cache(f_readmit, models$best_perf1$death, train_list, val_list, EVAL_TIMES)
cif_cache_bp2 <- .holdout_build_cif_cache(f_readmit, models$best_perf2$death, train_list, val_list, EVAL_TIMES)
results_boot_val_bp1_cif <- .holdout_inject_readmission_cif(results_boot_val_bp1, cif_cache_bp1)
results_boot_val_bp2_cif <- .holdout_inject_readmission_cif(results_boot_val_bp2, cif_cache_bp2)

cal_readmit_netrisk <- .holdout_calibrate_readmit_from_raw(results_boot_val_bp1, times = CAL_TIMES)
cal_readmit_netrisk_km <- .holdout_calibrate_readmit_from_raw(results_boot_val_bp1, times = CAL_TIMES, observed = "km")
cal_readmit_bp1     <- .holdout_calibrate_readmit_from_raw(results_boot_val_bp1_cif, times = CAL_TIMES)
cal_readmit_bp2     <- .holdout_calibrate_readmit_from_raw(results_boot_val_bp2_cif, times = CAL_TIMES)
cal_death_bp1       <- calibrate_death_holdout(models$best_perf1$death, train_list, val_list, times = CAL_TIMES)
cal_death_bp2       <- calibrate_death_holdout(models$best_perf2$death, train_list, val_list, times = CAL_TIMES)

In [ ]:
#| label: holdout-calibration-tables
source(file.path(project_root, "cons/_alt_scripts/holdout_calibration_panel.R"))
cal_list <- list(
  readmit_netrisk = cal_readmit_netrisk, readmit_netrisk_km= cal_readmit_netrisk_km, readmit_bp1_cif = cal_readmit_bp1, readmit_bp2_cif = cal_readmit_bp2,
  death_bp1 = cal_death_bp1, death_bp2 = cal_death_bp2
)
.holdout_calibration_summary_table(cal_list) |>
  knitr::kable("markdown", digits = 4,
    caption = "Held-out 20%: ICI / ECE / E:O by horizon and arm (pooled over imputations)")


In [ ]:
#| label: holdout-calibration-plot-3arm
p_cal_readmit <- .holdout_calibration_curve_plot_n(
  list(net = cal_readmit_netrisk, net_km = cal_readmit_netrisk_km, cifA = cal_readmit_bp1, cifB = cal_readmit_bp2),
  row_labels = c(net = "Net risk (1-S(t))\nvs AJ", net_km= "Net risk (1-S(t))\nvs KM", cifA = "Joint CIF x\nAll predictors", cifB = "Joint CIF x\nSHAP informed"),
  horizons = CAL_TIMES, lang = "en"
)
print(p_cal_readmit)
ggplot2::ggsave(file.path(figs_out, "holdout_calibration_readmit_3arm.png"), p_cal_readmit, width = 10, height = 7, dpi = 300)


In [ ]:
#| label: holdout-calibration-plot-3arm-es
p_cal_readmit_es <- .holdout_calibration_curve_plot_n(
  list(net = cal_readmit_netrisk, net_km = cal_readmit_netrisk_km, cifA = cal_readmit_bp1, cifB = cal_readmit_bp2),
  row_labels = c(net = "Riesgo neto(1-S(t))\nvs AJ", net_km= "Riesgo neto(1-S(t))\nvs KM", cifA = "Inc. acumulada x\nCompleto", cifB = "Inc. acumulada x\nInformado por SHAP"),
  horizons = CAL_TIMES, lang = "es"
)
print(p_cal_readmit_es)
ggplot2::ggsave(file.path(figs_out, "holdout_calibration_readmit_3arm_es.png"), p_cal_readmit, width = 10, height = 7, dpi = 300)


In [ ]:
#| label: holdout-calibration-plot
p_cal_death <- .holdout_calibration_curve_plot_n(
  list(bp1 = cal_death_bp1, bp2 = cal_death_bp2),
  row_labels = c(bp1 = "Death, best_perf1 (Full PH)", bp2 = "Death, best_perf2 (SHAP)"),
  horizons = CAL_TIMES, lang = "en"
)
print(p_cal_death)
ggplot2::ggsave(file.path(figs_out, "holdout_calibration_death_2arm.png"), p_cal_death, width = 9, height = 6, dpi = 300)


In [ ]:
#| label: holdout-cal-fig-epi
#| fig-width: 16
#| fig-height: 11
# Three-model Epidemiology calibration figure, one fair calibration per model.
# A = Readmission (JOINT CIF via CSC paired with primary mortality updated2; observed = AJ).
#     Net risk 1-S(t) deliberately not shown here (miscalibrated by construction vs AJ);
#     it lives in the supplement (.holdout_calibration_curve_plot_n).
# B = Mortality, all predictors (Full PH, best_perf1).  C = Mortality, SHAP-informed (best_perf2).
 .t0 <- Sys.time()
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/cal_panel_epi_functions.R"))
readmit_row <- .make_cal_panel(cal_readmit_bp1, "#2166AC", x_lim = c(0, 0.4),  x_by = 0.2)
death_A     <- .make_cal_panel(cal_death_bp1,   "#B2182B", x_lim = c(0, 0.15), x_by = 0.05)
death_B     <- .make_cal_panel(cal_death_bp2,   "#B2182B", x_lim = c(0, 0.15), x_by = 0.05)

  inner_theme <- ggplot2::theme(                                                                                                                                          
    axis.title.x = ggplot2::element_blank(),                                                                                                                              
    axis.title.y = ggplot2::element_blank()                                                                                                                               
  )                                                                                                                                                                                                                                                                                                                                            
  # Treat each four-panel row as a single element.                                                                                                                        
  rows <- patchwork::wrap_plots(                                                                                                                                          
    patchwork::wrap_elements(                                                                                                                                             
      full = readmit_row & inner_theme,                                                                                                                                   
      clip = FALSE                                                                                                                                                        
    ),                                                                                                                                                                    
    patchwork::wrap_elements(                                                                                                                                             
      full = death_A & inner_theme,                                                                                                                                       
      clip = FALSE                                                                                                                                                        
    ),                                                                                                                                                                    
    patchwork::wrap_elements(                                                                                                                                             
      full = death_B & inner_theme,                                                                                                                                       
      clip = FALSE                                                                                                                                                        
    ),                                                                                                                                                                    
    ncol = 1                                                                                                                                                              
  ) +                                                                                                                                                                     
    patchwork::plot_layout(ncol = 1) +                                                                                                                                    
    patchwork::plot_annotation(                                                                                                                                           
      tag_levels = "A",                                                                                                                                                   
      caption = "Predicted probability",                                                                                                                                  
      theme = ggplot2::theme(                                                                                                                                             
        plot.caption = ggplot2::element_text(                                                                                                                             
          size = 13,                                                                                                                                                      
          face = "bold",                                                                                                                                                  
          hjust = 0.5,                                                                                                                                                    
          family = tnr                                                                                                                                                    
        ),                                                                                                                                                                
        plot.tag = ggplot2::element_text(                                                                                                                                 
          face = "bold",                                                                                                                                                  
          size = 18,                                                                                                                                                      
          family = tnr                                                                                                                                                    
        )                                                                                                                                                                 
      )                                                                                                                                                                   
    )                                                                                                                                                                                                                                                                                                                                       
  # Add one global Y-axis title.                                                                                                                                          
  common_y_title <- patchwork::wrap_elements(                                                                                                                             
    full = grid::textGrob(                                                                                                                                                
      "Observed probability",                                                                                                                                             
      rot = 90,                                                                                                                                                           
      gp = grid::gpar(                                                                                                                                                    
        fontsize = 13,                                                                                                                                                    
        fontface = "bold",                                                                                                                                                
        fontfamily = tnr                                                                                                                                                  
      )                                                                                                                                                                   
    ),                                                                                                                                                                    
    clip = FALSE,                                                                                                                                                         
    ignore_tag = TRUE                                                                                                                                                     
  )                                                                                                                                                                                                                                                                                                                                               
  # Wrapping rows preserves its A–C tags and caption.                                                                                                                     
  final_three <- patchwork::wrap_plots(                                                                                                                                   
    common_y_title,                                                                                                                                                       
    patchwork::wrap_elements(                                                                                                                                             
      full = rows,                                                                                                                                                        
      clip = FALSE,                                                                                                                                                       
      ignore_tag = TRUE                                                                                                                                                   
    ),                                                                                                                                                                    
    ncol = 2,                                                                                                                                                             
    widths = c(0.035, 1)                                                                                                                                                  
  )                                                                                                                                                                                                                                                                                                                                               
  print(final_three)                                                                                                                                                                                                                                                                                                                             
  message(sprintf(                                                                                                                                                        
    "Elapsed time: %.2f minutes",                                                                                                                                         
    as.numeric(difftime(Sys.time(), .t0, units = "mins"))                                                                                                                 
  ))         
figratio<-.72
ggplot2::ggsave(file.path(figs_out, "holdout_calibration_three_models_AJ.tiff"),
                final_three, width = 35*figratio, height = 20*figratio, units = "cm", dpi = 600, compression = "lzw")
ggplot2::ggsave(file.path(figs_out, "holdout_calibration_three_models_AJ.png"),
                final_three, width = 35*figratio, height = 20*figratio, units = "cm", dpi = 600)


## Decision Curve Analysis (held-out 20%)

Reuses the audited engine `run_adca_from_results_boot` via `run_dca_full_summary` on the held-out `$raw_predictions`: **Aalen-Johansen** observed risk for readmission (death competing), **1−KM** for mortality. Net-benefit math unchanged from the original.

In [ ]:
#| label: holdout-dca-run-plot
source(file.path(project_root, "cons/_alt_scripts/make_dca_panel_figure_n.R"))
dca_models_full <- run_dca_full_summary(list(best_perf1 = results_boot_val_bp1, best_perf2 = results_boot_val_bp2), horizons = DCA_HORIZONS)
dca_readmit_3arm <- run_dca_full_summary(
  list(net_risk = results_boot_val_bp1, cif_bp1 = results_boot_val_bp1_cif, cif_bp2 = results_boot_val_bp2_cif),
  horizons = DCA_HORIZONS
)
dca_nb_bp1 <- summarize_dca_nb(dca_models_full$best_perf1$summary)
dca_nb_bp2 <- summarize_dca_nb(dca_models_full$best_perf2$summary)

p_dca_readmit_3arm <- make_dca_panel_figure_n(
  list(net_risk = dca_readmit_3arm$net_risk$summary, cif_bp1 = dca_readmit_3arm$cif_bp1$summary, cif_bp2 = dca_readmit_3arm$cif_bp2$summary),
  outcome = "readmission", horizons = DCA_HORIZONS,
  row_labels = c(net_risk = "Net risk (1-S(t))", cif_bp1 = "Joint CIF x updated2", cif_bp2 = "Joint CIF x rule2")
)
print(p_dca_readmit_3arm)
ggplot2::ggsave(file.path(figs_out, "holdout_dca_readmit_3arm.png"), p_dca_readmit_3arm, width = 11, height = 8, dpi = 300)

In [ ]:
#| label: holdout-dca-table
knitr::kable(dca_nb_bp1$any_useful, "markdown", digits = 4,
  caption = "best_perf1 (held-out 20%): does the model beat treat-all & treat-none at any threshold?")


In [ ]:
#| label: holdout-dca-plot
# Row A = best_perf1 (Full PH death), Row B = best_perf2 (SHAP death)
make_dca_panel_figure(dca_models_full$best_perf1$summary,
                      dca_models_full$best_perf2$summary,
                      outcome = "readmission", horizons = c(6, 12, 36, 60))
make_dca_panel_figure(dca_models_full$best_perf1$summary,
                      dca_models_full$best_perf2$summary,
                      outcome = "death", horizons = c(6, 12, 36, 60))


In [ ]:
#| label: holdout-dca-panel-figures-std
#| message: false
# Standardized-net-benefit DCA panels (held-out 20%) via the shared .R helpers.
# Standardized NB = net benefit / observed event rate at each horizon (1.0 = treat-all at threshold 0).
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/make_dca_panel_figure_std.R"))
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/make_dca_panel_figure_n.R"))
stopifnot(exists("dca_models_full"), exists("dca_readmit_3arm"))
DCA_HZ_STD <- c(6, 12, 36, 60)

# READMISSION: the three predicted-risk arms (net risk 1-S(t), joint CIF x updated2, x rule2),
# NOT the old degenerate best_perf1-vs-best_perf2 (which were identical, shared readmit model).
p_readmit_panel_hold <- make_dca_panel_figure_n(
  list(net_risk = dca_readmit_3arm$net_risk$summary,
       cif_bp1  = dca_readmit_3arm$cif_bp1$summary,
       cif_bp2  = dca_readmit_3arm$cif_bp2$summary),
  outcome = "readmission", horizons = DCA_HZ_STD, standardized = TRUE, lang = "en",
  row_labels = c(net_risk = "Net risk (1-S(t))",
                 cif_bp1  = "Joint CIF x updated2",
                 cif_bp2  = "Joint CIF x rule2"))
print(p_readmit_panel_hold)

# DEATH: unchanged (Full PH vs SHAP)
p_death_panel_hold <- make_dca_panel_figure_std(
  summary_full = dca_models_full$best_perf1$summary,
  summary_ml   = dca_models_full$best_perf2$summary,
  outcome = "death", horizons = DCA_HZ_STD, lang = "en",
  row_labels = c(A = "All\npredictors", B = "SHAP-\ninformed"))
print(p_death_panel_hold)

# Combined A/B/C: A = readmission (FAIR joint CIF x updated2, not net risk), B = mortality all
# predictors (Full PH), C = mortality SHAP-informed. The abc helper reads readmission + death B
# from best_perf1 and death C from best_perf2, so we feed it the CIF object as best_perf1.
dca_abc_input <- list(best_perf1 = dca_readmit_3arm$cif_bp1,   # readmission = CIF x updated2 ; death = Full PH
                      best_perf2 = dca_models_full$best_perf2) # death = SHAP
p_dca_abc <- make_dca_panel_figure_abc(
  dca_abc_input, horizons = DCA_HZ_STD, lang = "en", tnr = "Times New Roman")
print(p_dca_abc)

fig_dir <- file.path(if (exists("project_root")) project_root else getwd(), "cons", "_figs")
if (!dir.exists(fig_dir)) dir.create(fig_dir, recursive = TRUE)
for (nm in c("p_dca_holdout_readmit_3arm", "p_dca_holdout_death_bp1_vs_bp2", "p_dca_holdout_abc")) {
  obj <- switch(nm,
    p_dca_holdout_readmit_3arm     = p_readmit_panel_hold,
    p_dca_holdout_death_bp1_vs_bp2 = p_death_panel_hold,
    p_dca_holdout_abc              = p_dca_abc)
  wd <- if (nm == "p_dca_holdout_abc") 26 else 17.8 * 1.2
  ht <- if (nm == "p_dca_holdout_abc") 18 else 12 * 1.2
  ggplot2::ggsave(file.path(fig_dir, paste0(nm, ".png")), obj, width = wd, height = ht, units = "cm", dpi = 600)
  ggplot2::ggsave(file.path(fig_dir, paste0(nm, ".pdf")), obj, width = wd, height = ht, units = "cm",
                  device = grDevices::cairo_pdf)
}
message(attr(p_dca_abc, "caption"))


In [ ]:
#| label: holdout-dca-per1000
#| message: false
# Absolute "per 1000" translation of the HELD-OUT DCA, with replicate CIs.
.per1000_holdout <- function(sm, rk, thr_show, horizons = c(12, 36, 60)) {
  d <- sm[sm$risk == rk & sm$strategy == "Model" & sm$horizon %in% horizons, , drop = FALSE]
  f1 <- function(v) formatC(round(v, 1), format = "f", digits = 1)
  ci <- function(m, lo, hi) sprintf("%s (%s, %s)", f1(m), f1(lo), f1(hi))
  out <- do.call(rbind, lapply(thr_show, function(t) {
    r <- d[abs(d$threshold - t) < 1e-9, , drop = FALSE]; if (!nrow(r)) return(NULL)
    data.frame(
      horizon = r$horizon, thr = paste0(t * 100, "%"),
      in_focus = ifelse(t >= r$focus_lower & t <= r$focus_upper, "*", ""),
      event_per1000 = f1(r$observed_event_risk_mean * 1000),
      captured_vs_none = ci(r$net_benefit_mean*1000, r$net_benefit_q025*1000, r$net_benefit_q975*1000),
      avoided_vs_all  = ci(r$interventions_avoided_mean*10, r$interventions_avoided_q025*10, r$interventions_avoided_q975*10),
      stringsAsFactors = FALSE)
  }))
  out[order(out$horizon), ]
}
thr_re <- c(0.05, 0.10, 0.15, 0.20, 0.30)
thr_de <- c(0.01, 0.02, 0.03, 0.05, 0.10)

# READMISSION: three predicted-risk arms (same shared Cox model; only the predicted risk differs)
cat("== READMISSION arm 1: net risk 1-S(t) (per 1000) ==\n")
print(.per1000_holdout(dca_readmit_3arm$net_risk$summary, "readmission", thr_re), row.names = FALSE)
cat("\n== READMISSION arm 2: joint CIF (CSC) x updated2 death (per 1000) ==\n")
print(.per1000_holdout(dca_readmit_3arm$cif_bp1$summary, "readmission", thr_re), row.names = FALSE)
cat("\n== READMISSION arm 3: joint CIF (CSC) x rule2 death (per 1000) ==\n")
print(.per1000_holdout(dca_readmit_3arm$cif_bp2$summary, "readmission", thr_re), row.names = FALSE)

# DEATH: unchanged (two mortality models)
cat("\n== DEATH A = best_perf1 (Full PH) per 1000 ==\n")
print(.per1000_holdout(dca_models_full$best_perf1$summary, "death", thr_de), row.names = FALSE)
cat("\n== DEATH B = best_perf2 (SHAP) per 1000 ==\n")
print(.per1000_holdout(dca_models_full$best_perf2$summary, "death", thr_de), row.names = FALSE)

cat("\ncaptured_vs_none = net true cases caught /1000 (vs treat-none);",
    "avoided_vs_all = unnecessary interventions avoided /1000 (vs treat-all).",
    "Readmission arm 1 is 1-S(t) (net risk); arms 2-3 are the competing-risk CIF via CSC.\n")


### Bootstrap


In [ ]:
#| label: holdout-dca-bootstrap-per1000-2
#| message: true
#| warning: false
#| results: asis
# ---- Purpose ----------------------------------------------------------------
# This chunk performs patient-level bootstrap DCA for all readmission and mortality prediction specifications.
# Readmission includes cause-specific net risk and both competing-risk CIF constructions.
# Mortality includes the Full PH and SHAP-informed Cox models.
# Treat-all and treat-none are retained as reference strategies.
# Models and individual predictions remain frozen. Only held-out patients are resampled.

# survival is required by the sourced DCA helpers (Kaplan-Meier / Aalen-Johansen
# estimators of the observed event risk used inside the net-benefit calculation).
suppressPackageStartupMessages(library(survival))
# Guard: stop early with a clear message if the notebook setup chunks were not run.
if (!exists("project_root")) stop("project_root is missing. Run the notebook setup chunks first.", call. = FALSE)
# out_dir: where the CSV tables are exported (falls back to cons/_out if unset upstream).
if (!exists("out_dir")) out_dir <- file.path(project_root, "cons", "_out")
# data_out: where the archived RDS bundle is intended to live.
data_out <- file.path(project_root, "data", "20241015_out")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(data_out, recursive = TRUE, showWarnings = FALSE)
# Frozen holdout prediction objects produced by the upstream validation chunks:
# bp1 = Full PH specification, bp2 = SHAP-informed specification,
# *_cif = competing-risk cumulative-incidence variants of the readmission predictions.
required_objects <- c("results_boot_val_bp1", "results_boot_val_bp2", "results_boot_val_bp1_cif", "results_boot_val_bp2_cif")
missing_objects <- required_objects[!vapply(required_objects, exists, logical(1), inherits = TRUE)]
if (length(missing_objects) > 0L) stop("Missing prediction objects: ", paste(missing_objects, collapse = ", "), ".", call. = FALSE)
# Helper script providing .extract_replicates() (pulls replicate blocks out of the
# results objects) and .compute_replicate_curves() (computes net-benefit curves
# for a single prediction block over thresholds x horizons).
DCA_ADCA_SOURCE <- file.path(project_root, "cons/_alt_scripts/adca_from_results_boot.R")
source(DCA_ADCA_SOURCE)
# ---- Bootstrap configuration --------------------------------------------------
DCA_BOOT_B <- 1000L                      # number of patient-level bootstrap resamples
DCA_BOOT_SEED <- 2125L                   # seed making the resample indices reproducible
DCA_BOOT_HORIZONS <- c(6, 12, 36, 60)    # horizons (months) at which net benefit is evaluated
DCA_BOOT_DENSE_GRID <- TRUE              # TRUE: dense threshold grid for curves; FALSE: few reference thresholds
# Readmission thresholds span 1%-40% (higher expected event rates).
DCA_THR_READMISSION <- if (DCA_BOOT_DENSE_GRID) seq(0.01, 0.40, by = 0.01) else c(0.05, 0.10, 0.15, 0.20, 0.30)
# Mortality thresholds span 0.5%-10% (death is rarer, so clinically relevant thresholds are lower).
DCA_THR_MORTALITY <- if (DCA_BOOT_DENSE_GRID) seq(0.005, 0.10, by = 0.005) else c(0.005, 0.01, 0.02, 0.03, 0.05, 0.10)
# ---- Parallel backend ---------------------------------------------------------
# Use future + future.apply only if all three packages are available.
DCA_USE_PARALLEL <- requireNamespace("future", quietly = TRUE) && requireNamespace("future.apply", quietly = TRUE) && requireNamespace("parallelly", quietly = TRUE)
if (DCA_USE_PARALLEL) {
  # Make PSOCK worker startup deterministic and patient: sequential node setup and
  # a long connect timeout avoid spurious failures on slow machines; raise the
  # globals size limit because each worker receives the pooled prediction block.
  options(parallelly.makeNodePSOCK.setup_strategy = "sequential")
  options(parallelly.makeNodePSOCK.connectTimeout = 600)
  options(future.globals.maxSize = 10 * 1024^3)
  # Use at most half the available cores, capped at 12, always at least 1.
  DCA_N_WORKERS <- min(12L, max(1L, parallelly::availableCores() %/% 2L))
  # Try to start a multisession plan; on any error fall back to serial execution.
  parallel_ok <- tryCatch({
    future::plan(future::multisession, workers = DCA_N_WORKERS)
    TRUE
  }, error = function(e) {
    message("Parallel setup failed: ", conditionMessage(e))
    FALSE
  })
  if (!parallel_ok) {
    DCA_USE_PARALLEL <- FALSE
    future::plan(future::sequential)
  }
}
message("DCA bootstrap workers: ", if (DCA_USE_PARALLEL) DCA_N_WORKERS else 0L)
# ---- Helper: pool replicate blocks --------------------------------------------
# Each results_boot_val_* object contains several replicate blocks over the same
# held-out patients. Pool them by averaging the patient-level predicted-risk
# matrices element-wise (all blocks must have identical row counts), keeping the
# first block's structure (times, events, competing-risk fields) with replicate_id
# reset to 1 to mark it as the single pooled block.
.dca_pool_blocks <- function(blocks) {
  nr <- nrow(blocks[[1]]$pred_risk)
  stopifnot(all(vapply(blocks, function(b) nrow(b$pred_risk) == nr, logical(1))))
  b0 <- blocks[[1]]
  b0$pred_risk <- Reduce("+", lapply(blocks, function(b) b$pred_risk)) / length(blocks)
  b0$replicate_id <- 1L
  b0
}
# ---- Helper: bootstrap DCA for one model specification ------------------------
# block: pooled predictions plus observed times/events for the held-out patients.
# thresholds/horizons: evaluation grid. observed_method: how the observed event
#   risk is estimated ("aalen-johansen" for readmission with death as competing
#   risk, "km" for mortality). model_id/model_label/prediction_scale: metadata
#   carried into the output. B: number of resamples. seed: resampling seed.
.dca_boot_one_arm <- function(block, thresholds, horizons, observed_method, model_id, model_label, prediction_scale, B, seed) {
  message(sprintf("Running DCA bootstrap for %s with B = %d.", model_label, B))
  thresholds <- sort(unique(thresholds))
  n <- nrow(block$pred_risk)
  # Columns retained from the point-estimate curves before merging with CIs.
  keep <- c("horizon", "threshold", "n_obs", "positive_rate", "observed_event_risk", "nb_model", "nb_treat_all", "nb_treat_none", "std_nb_model", "std_nb_treat_all", "interventions_avoided")
  # Point estimates: DCA curves on the full (un-resampled) holdout set.
  point_results <- .compute_replicate_curves(block, thresholds, horizons, readmit_method = observed_method)
  if (!nrow(point_results)) stop("No DCA point estimates were produced for ", model_id, ".", call. = FALSE)
  point_results <- point_results[, keep, drop = FALSE]
  # Key "horizon_threshold" used to align bootstrap outputs with point-estimate rows.
  point_key <- paste(point_results$horizon, point_results$threshold, sep = "_")
  # Draw all B resample index vectors upfront under a fixed seed; the same
  # resamples are then used whether execution is serial or parallel.
  set.seed(seed)
  bootstrap_indices <- lapply(seq_len(B), function(i) sample.int(n, n, replace = TRUE))
  # One bootstrap replicate: subset the block to the resampled patients
  # (predicted risks, follow-up times, event indicators, and competing-risk
  # time/status when present), recompute the curves, and return named vectors
  # keyed by horizon_threshold for later alignment.
  one_replicate <- function(index) {
    # Parallel workers start with a fresh session: reload survival and the
    # sourced helpers there if they are missing.
    if (!exists(".compute_replicate_curves", mode = "function")) {
      suppressPackageStartupMessages(library(survival))
      source(DCA_ADCA_SOURCE)
    }
    b <- block
    b$pred_risk <- b$pred_risk[index, , drop = FALSE]
    b$time <- b$time[index]
    b$event <- b$event[index]
    if (!is.null(b$cr_ftime)) b$cr_ftime <- b$cr_ftime[index]
    if (!is.null(b$cr_fstatus)) b$cr_fstatus <- b$cr_fstatus[index]
    d <- .compute_replicate_curves(b, thresholds, horizons, readmit_method = observed_method)
    k <- paste(d$horizon, d$threshold, sep = "_")
    list(nb_model = setNames(d$nb_model, k), nb_treat_all = setNames(d$nb_treat_all, k), std_nb_model = setNames(d$std_nb_model, k), interventions_avoided = setNames(d$interventions_avoided, k), observed_event_risk = setNames(d$observed_event_risk, k), positive_rate = setNames(d$positive_rate, k))
  }
  # Evaluate all B replicates, in parallel when the backend is available.
  replicates <- if (DCA_USE_PARALLEL) {
    future.apply::future_lapply(bootstrap_indices, one_replicate, future.seed = TRUE)
  } else {
    lapply(bootstrap_indices, one_replicate)
  }
  # Extract one field across all replicates as a matrix with rows aligned to
  # point_key and one column per bootstrap replicate; guard the degenerate
  # case of a single grid point where sapply would drop the dimension.
  pull_matrix <- function(field) {
    m <- sapply(replicates, function(z) z[[field]][point_key])
    if (is.null(dim(m))) m <- matrix(m, nrow = length(point_key))
    m
  }
  nb_model_boot <- pull_matrix("nb_model")
  nb_all_boot <- pull_matrix("nb_treat_all")
  std_model_boot <- pull_matrix("std_nb_model")
  avoided_boot <- pull_matrix("interventions_avoided")
  event_boot <- pull_matrix("observed_event_risk")
  positive_boot <- pull_matrix("positive_rate")
  # Row-wise percentile of the bootstrap distribution -> CI bounds.
  qrow <- function(m, probability) apply(m, 1, quantile, probs = probability, na.rm = TRUE, names = FALSE)
  # Assemble the output table: point estimates plus 2.5%/97.5% percentile CIs.
  out <- data.frame(
    model_id = model_id, model_label = model_label, outcome = block$risk,
    prediction_scale = prediction_scale,
    # Name of the observed-risk estimator, for traceability.
    observed_method = ifelse(block$risk == "readmission", "Aalen-Johansen", "Kaplan-Meier"),
    horizon = point_results$horizon, threshold = point_results$threshold, n = point_results$n_obs,
    # Focus window: thresholds within [0.5x, 2x] the observed event risk,
    # i.e. the clinically plausible range around the outcome prevalence.
    in_focus = point_results$threshold >= 0.5 * point_results$observed_event_risk & point_results$threshold <= 2 * point_results$observed_event_risk,
    event_risk = point_results$observed_event_risk,
    event_risk_lo = qrow(event_boot, 0.025), event_risk_hi = qrow(event_boot, 0.975),
    positive_rate = point_results$positive_rate,
    positive_rate_lo = qrow(positive_boot, 0.025), positive_rate_hi = qrow(positive_boot, 0.975),
    # Net benefit of the model and of the treat-all strategy; treat-none is 0 by definition.
    nb_model = point_results$nb_model,
    nb_model_lo = qrow(nb_model_boot, 0.025), nb_model_hi = qrow(nb_model_boot, 0.975),
    nb_treat_all = point_results$nb_treat_all,
    nb_treat_all_lo = qrow(nb_all_boot, 0.025), nb_treat_all_hi = qrow(nb_all_boot, 0.975),
    nb_treat_none = 0,
    # Standardized net benefit (net benefit divided by observed event risk).
    std_nb_model = point_results$std_nb_model,
    std_nb_model_lo = qrow(std_model_boot, 0.025), std_nb_model_hi = qrow(std_model_boot, 0.975),
    # Per-1,000-patient scalings for clinical interpretability.
    net_cases_per1000 = point_results$nb_model * 1000,
    net_cases_per1000_lo = qrow(nb_model_boot, 0.025) * 1000,
    net_cases_per1000_hi = qrow(nb_model_boot, 0.975) * 1000,
    # interventions_avoided is expressed per 100 patients by convention; x10 -> per 1,000.
    avoided_per1000 = point_results$interventions_avoided * 10,
    avoided_per1000_lo = qrow(avoided_boot, 0.025) * 10,
    avoided_per1000_hi = qrow(avoided_boot, 0.975) * 10,
    B = B,
    # Number of bootstrap replicates yielding a finite nb_model for this grid point.
    B_valid = rowSums(is.finite(nb_model_boot)),
    seed = seed, stringsAsFactors = FALSE
  )
  message(sprintf("Completed %s: %d threshold-horizon combinations.", model_label, nrow(out)))
  out
}
# ---- Build the five pooled prediction blocks -----------------------------------
# Readmission, cause-specific net risk 1 - S(t), from the Full PH specification,
# with death attached as the competing event for the observed-risk estimator.
readmit_netrisk_pool <- .dca_pool_blocks(.extract_replicates(results_boot_val_bp1, "readmission", attach_competing = TRUE, competing_risk = "death"))
# Readmission CIF predictions with mortality from the Full PH model.
readmit_cif_fullph_pool <- .dca_pool_blocks(.extract_replicates(results_boot_val_bp1_cif, "readmission", attach_competing = TRUE, competing_risk = "death"))
# Readmission CIF predictions with mortality from the SHAP-informed model.
readmit_cif_shap_pool <- .dca_pool_blocks(.extract_replicates(results_boot_val_bp2_cif, "readmission", attach_competing = TRUE, competing_risk = "death"))
# Mortality predictions from the Full PH and SHAP-informed specifications.
death_fullph_pool <- .dca_pool_blocks(.extract_replicates(results_boot_val_bp1, "death"))
death_shap_pool <- .dca_pool_blocks(.extract_replicates(results_boot_val_bp2, "death"))
# Sanity check: all five pools must cover the same held-out patients.
pooled_sizes <- c(nrow(readmit_netrisk_pool$pred_risk), nrow(readmit_cif_fullph_pool$pred_risk), nrow(readmit_cif_shap_pool$pred_risk), nrow(death_fullph_pool$pred_risk), nrow(death_shap_pool$pred_risk))
stopifnot(length(unique(pooled_sizes)) == 1L)
# ---- Run the bootstrap for all five specifications ------------------------------
dca_boot_all <- rbind(
  .dca_boot_one_arm(readmit_netrisk_pool, DCA_THR_READMISSION, DCA_BOOT_HORIZONS, "aalen-johansen", "readmit::netrisk", "Readmission net risk 1-S(t)", "Cause-specific net risk", DCA_BOOT_B, DCA_BOOT_SEED),
  .dca_boot_one_arm(readmit_cif_fullph_pool, DCA_THR_READMISSION, DCA_BOOT_HORIZONS, "aalen-johansen", "readmit::bp1_cif", "Readmission CIF with Full PH mortality", "Competing-risk CIF", DCA_BOOT_B, DCA_BOOT_SEED),
  .dca_boot_one_arm(readmit_cif_shap_pool, DCA_THR_READMISSION, DCA_BOOT_HORIZONS, "aalen-johansen", "readmit::bp2_cif", "Readmission CIF with SHAP-informed mortality", "Competing-risk CIF", DCA_BOOT_B, DCA_BOOT_SEED),
  .dca_boot_one_arm(death_fullph_pool, DCA_THR_MORTALITY, DCA_BOOT_HORIZONS, "km", "death::best_perf1", "Mortality Full PH", "Predicted mortality risk", DCA_BOOT_B, DCA_BOOT_SEED),
  .dca_boot_one_arm(death_shap_pool, DCA_THR_MORTALITY, DCA_BOOT_HORIZONS, "km", "death::best_perf2", "Mortality SHAP-informed", "Predicted mortality risk", DCA_BOOT_B, DCA_BOOT_SEED)
)
rownames(dca_boot_all) <- NULL
# Every grid point must have B valid (finite) bootstrap values.
stopifnot(all(dca_boot_all$B_valid == DCA_BOOT_B))
# Restrict to the prespecified focus windows (thresholds near the observed event risk).
dca_boot_focus <- dca_boot_all[dca_boot_all$in_focus, , drop = FALSE]
# ---- Display table ---------------------------------------------------------------
# Format "point (lo to hi)" at one decimal, and thresholds as percentages
# (one decimal below 1%, integer otherwise).
.dca_ci <- function(point, lower, upper) sprintf("%.1f (%.1f to %.1f)", point, lower, upper)
.dca_threshold_label <- function(x) ifelse(x < 0.01, sprintf("%.1f%%", 100 * x), sprintf("%.0f%%", 100 * x))
dca_boot_display <- data.frame(
  Model = dca_boot_focus$model_label,
  Horizon = dca_boot_focus$horizon,
  Threshold = .dca_threshold_label(dca_boot_focus$threshold),
  `Observed events per 1,000 (95% CI)` = .dca_ci(dca_boot_focus$event_risk * 1000, dca_boot_focus$event_risk_lo * 1000, dca_boot_focus$event_risk_hi * 1000),
  `Net true cases per 1,000 vs treat-none (95% CI)` = .dca_ci(dca_boot_focus$net_cases_per1000, dca_boot_focus$net_cases_per1000_lo, dca_boot_focus$net_cases_per1000_hi),
  `Unnecessary interventions avoided per 1,000 vs treat-all (95% CI)` = .dca_ci(dca_boot_focus$avoided_per1000, dca_boot_focus$avoided_per1000_lo, dca_boot_focus$avoided_per1000_hi),
  check.names = FALSE
)
#print(knitr::kable(dca_boot_display, format = "markdown", align = c("l", "c", "c", "c", "c", "c"), caption = "Patient-bootstrap decision-curve results within the prespecified focus windows."))
# ---- Exports ---------------------------------------------------------------------
utils::write.csv(dca_boot_all, file.path(out_dir, "pred23_holdout_dca_bootstrap_all_models.csv"), row.names = FALSE)
utils::write.csv(dca_boot_focus, file.path(out_dir, "pred23_holdout_dca_bootstrap_focus_windows.csv"), row.names = FALSE)
# Bundle results + full configuration for archival (note: net-risk thresholds are
# cause-specific net risks, not competing-risk absolute probabilities).
dca_boot_archive <- list(results = dca_boot_all, focus_windows = dca_boot_focus, config = list(B = DCA_BOOT_B, seed = DCA_BOOT_SEED, horizons = DCA_BOOT_HORIZONS, thresholds_readmission = DCA_THR_READMISSION, thresholds_mortality = DCA_THR_MORTALITY, dense_grid = DCA_BOOT_DENSE_GRID, readmission_observed_method = "Aalen-Johansen", mortality_observed_method = "Kaplan-Meier", note = "Net-risk thresholds are cause-specific net-risk thresholds and are not competing-risk absolute probabilities."))
saveRDS(
  dca_boot_archive,
  file.path(data_out, "pred23_holdout_dca_bootstrap_all_models.rds")
)
# Reset the future backend so later chunks run sequentially.
if (DCA_USE_PARALLEL) future::plan(future::sequential)
message("DCA bootstrap completed for all five model specifications.")
message("Full CSV: ", file.path(out_dir, "pred23_holdout_dca_bootstrap_all_models.csv"))
message("Focus-window CSV: ", file.path(out_dir, "pred23_holdout_dca_bootstrap_focus_windows.csv"))
message("RDS: ", file.path(data_out, "pred23_holdout_dca_bootstrap_all_models.rds"))

~129 minutes

In [ ]:
#| label: holdout-dca-publication-figure
#| fig-width: 7.2
#| fig-height: 7.6
#| fig-cap: "Decision-curve analysis in the held-out sample. Curves show net benefit, expressed as net true cases per 1,000 persons. Mortality ribbons are model-specific pointwise 95% patient-bootstrap confidence intervals. In the readmission panels, the gray ribbon spans the model-specific pointwise 95% bootstrap intervals, which were nearly coincident. The dot-dashed reference denotes treat all and the horizontal reference at zero denotes treat none. Only the nonnegative segment of the treat-all strategy is displayed. Readmission CIFs account for death as a competing event. The net-risk readmission curve is included as an audit because its thresholds are not competing-risk absolute probabilities. Axis ranges vary by horizon."
#| warning: false
#| message: false
# Require the packages used to construct and export the publication figure.
stopifnot(requireNamespace("ggplot2", quietly = TRUE), requireNamespace("dplyr", quietly = TRUE), requireNamespace("patchwork", quietly = TRUE), requireNamespace("cowplot", quietly = TRUE), requireNamespace("scales", quietly = TRUE))
# EDIT ONLY THE TEXT VALUES IN THIS NAMED VECTOR; every displayed label is derived from it.
dca_labels <- c(death_all = "Mortality:\nCIF, All\npredictors", death_shap = "Mortality:\nCIF, SHAP-\ninformed", read_netrisk = "Readmission:\nNet risk 1-S(t)", read_all = "Readmission:\n+CIF, All\npredictors", read_shap = "Readmission:\n+CIF, SHAP-\ninformed", treat_all = "Reference:\nTreat all", treat_none = "Reference:\nTreat none")
# Keep stable internal keys separate from display labels so wording or line-break changes cannot break mappings.
dca_model_keys <- c("death_all", "death_shap", "read_netrisk", "read_all", "read_shap")
dca_reference_keys <- c("treat_all", "treat_none")
dca_legend_keys <- c(dca_model_keys, dca_reference_keys)
stopifnot(identical(names(dca_labels), dca_legend_keys))
# Map the original source labels to stable internal keys; do not add display text here.
dca_model_key_map <- c("Mortality Full PH" = "death_all", "Mortality SHAP-informed" = "death_shap", "Readmission net risk 1-S(t)" = "read_netrisk", "Readmission CIF with Full PH mortality" = "read_all", "Readmission CIF with SHAP-informed mortality" = "read_shap")
# Load the dense DCA results only when they are not already available in memory.
if (!exists("dca_boot_all", inherits = TRUE)) {
  dca_rds_candidates <- c("G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/pred23_holdout_dca_bootstrap_all_models.rds", "G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/pred23_holdout_validation_2026_07_17.rds")
  dca_rds_path <- dca_rds_candidates[file.exists(dca_rds_candidates)][1]
  if (is.na(dca_rds_path)) stop("No dense DCA RDS was found.")
  dca_loaded <- readRDS(dca_rds_path)
  if (is.data.frame(dca_loaded)) dca_boot_all <- dca_loaded else if (is.list(dca_loaded) && is.data.frame(dca_loaded$results)) dca_boot_all <- dca_loaded$results else if (is.list(dca_loaded) && is.data.frame(dca_loaded$dca_boot_all)) dca_boot_all <- dca_loaded$dca_boot_all else stop("The selected RDS does not contain a recognizable dense DCA result table.")
}
# Verify the variables used by the curves, confidence intervals, and reference strategies.
dca_required <- c("model_id", "model_label", "outcome", "horizon", "threshold", "in_focus", "event_risk", "net_cases_per1000", "net_cases_per1000_lo", "net_cases_per1000_hi", "B_valid")
stopifnot(all(dca_required %in% names(dca_boot_all)), all(dca_boot_all$B_valid == 1000L))
# Reproduce the descriptive windows and map source model labels to stable internal keys.
dca_plot_data <- dca_boot_all |>
  dplyr::filter(in_focus) |>
  dplyr::mutate(model_key = unname(dca_model_key_map[model_label]), horizon_label = factor(paste0(horizon, " months"), levels = paste0(c(6, 12, 36, 60), " months")))
if (anyNA(dca_plot_data$model_key)) stop("At least one source model_label is not registered in dca_model_key_map.")
dca_plot_data$model_key <- factor(dca_plot_data$model_key, levels = dca_model_keys)
# Define aesthetics using stable keys; these vectors do not change when dca_labels is edited.
dca_colors <- c(death_all = "#24455E", death_shap = "#8A5A2B", read_netrisk = "#8A8A8A", read_all = "#202020", read_shap = "#626262")
dca_fill_colors <- c(death_all = "#9FB9CA", death_shap = "#D5B287", read_netrisk = "#D9D9D9", read_all = "#D9D9D9", read_shap = "#D9D9D9")
dca_linetypes <- c(death_all = "solid", death_shap = "longdash", read_netrisk = "dotted", read_all = "solid", read_shap = "longdash")
dca_legend_colors <- c(dca_colors, treat_all = "#333333", treat_none = "#777777")
dca_legend_linetypes <- c(dca_linetypes, treat_all = "dotdash", treat_none = "solid")
stopifnot(identical(names(dca_legend_colors), dca_legend_keys), identical(names(dca_legend_linetypes), dca_legend_keys))
# Create one envelope spanning the nearly coincident readmission confidence intervals.
dca_readmission_envelope <- dca_plot_data |>
  dplyr::filter(outcome == "readmission") |>
  dplyr::group_by(horizon_label, threshold) |>
  dplyr::summarise(lower = min(net_cases_per1000_lo), upper = max(net_cases_per1000_hi), .groups = "drop")
# Construct smooth treat-all curves from the observed event risk in each displayed threshold window.
dca_reference_specs <- dca_plot_data |>
  dplyr::group_by(outcome, horizon_label) |>
  dplyr::summarise(event_risk = dplyr::first(event_risk), threshold_min = min(threshold), threshold_max = max(threshold), .groups = "drop")
dca_treat_all <- do.call(rbind, lapply(seq_len(nrow(dca_reference_specs)), function(i) {
  spec <- dca_reference_specs[i, , drop = FALSE]
  threshold_grid <- seq(spec$threshold_min, spec$threshold_max, length.out = 201L)
  data.frame(outcome = spec$outcome, horizon_label = spec$horizon_label, threshold = threshold_grid, net_cases_per1000 = 1000 * (spec$event_risk - threshold_grid) / (1 - threshold_grid))
})) |>
  dplyr::filter(net_cases_per1000 >= 0)
# Format threshold axes without losing the 0.5-percentage-point mortality increments.
.dca_percent_axis <- function(x) {
  pct <- 100 * x
  ifelse(abs(pct - round(pct)) < 1e-8, paste0(sprintf("%.0f", pct), "%"), paste0(sprintf("%.1f", pct), "%"))
}
# Apply a restrained journal-style theme while suppressing panel-specific legends.
dca_theme <- ggplot2::theme_classic(base_size = 8.5, base_family = "sans") +
  ggplot2::theme(plot.title = ggplot2::element_text(size = 9.5, face = "bold", hjust = 0), strip.background = ggplot2::element_blank(), strip.text = ggplot2::element_text(size = 8.2, face = "bold"), axis.title = ggplot2::element_text(size = 8.5), axis.text = ggplot2::element_text(size = 7.4, color = "black"), axis.line = ggplot2::element_line(linewidth = 0.35, color = "black"), axis.ticks = ggplot2::element_line(linewidth = 0.35, color = "black"), panel.spacing = grid::unit(8, "pt"), legend.position = "none", plot.margin = ggplot2::margin(4, 5, 2, 5))
# Generate one outcome panel with model curves, bootstrap uncertainty, and reference strategies.
.make_dca_panel <- function(outcome_value, panel_title) {
  panel_data <- dplyr::filter(dca_plot_data, outcome == outcome_value)
  panel_ref <- dplyr::filter(dca_treat_all, outcome == outcome_value) |>
    dplyr::mutate(reference_key = factor("treat_all", levels = dca_legend_keys))
  panel_none <- data.frame(yintercept = 0, reference_key = factor("treat_none", levels = dca_legend_keys))
  p <- ggplot2::ggplot(panel_data, ggplot2::aes(x = threshold, y = net_cases_per1000, color = model_key, linetype = model_key, group = model_key)) +
    ggplot2::geom_hline(data = panel_none, ggplot2::aes(yintercept = yintercept, color = reference_key, linetype = reference_key), inherit.aes = FALSE, linewidth = 0.35) +
    ggplot2::geom_line(data = panel_ref, ggplot2::aes(x = threshold, y = net_cases_per1000, color = reference_key, linetype = reference_key, group = 1), inherit.aes = FALSE, linewidth = 0.4)
  if (outcome_value == "readmission") {
    p <- p +
      ggplot2::geom_ribbon(data = dca_readmission_envelope, ggplot2::aes(x = threshold, ymin = lower, ymax = upper, group = 1), inherit.aes = FALSE, fill = "#CFCFCF", color = "#A8A8A8", linewidth = 0.20, alpha = 0.55)
  } else {
    p <- p +
      ggplot2::geom_ribbon(ggplot2::aes(ymin = net_cases_per1000_lo, ymax = net_cases_per1000_hi, fill = model_key, group = model_key), color = NA, alpha = 0.30, show.legend = FALSE) +
      ggplot2::scale_fill_manual(values = dca_fill_colors, limits = dca_model_keys, breaks = dca_model_keys, labels = unname(dca_labels[dca_model_keys]), drop = FALSE, name = NULL)
  }
  p +
    ggplot2::geom_line(linewidth = 0.75) +
    ggplot2::facet_wrap(~horizon_label, nrow = 1, scales = "free") +
    ggplot2::scale_color_manual(values = dca_legend_colors, limits = dca_legend_keys, breaks = dca_legend_keys, labels = unname(dca_labels[dca_legend_keys]), drop = FALSE, name = NULL) +
    ggplot2::scale_linetype_manual(values = dca_legend_linetypes, limits = dca_legend_keys, breaks = dca_legend_keys, labels = unname(dca_labels[dca_legend_keys]), drop = FALSE, name = NULL) +
    ggplot2::scale_x_continuous(labels = .dca_percent_axis, breaks = scales::breaks_pretty(n = 4), expand = ggplot2::expansion(mult = c(0.01, 0.03))) +
    ggplot2::scale_y_continuous(labels = scales::label_number(accuracy = 1), breaks = scales::breaks_pretty(n = 5), expand = ggplot2::expansion(mult = c(0.04, 0.08))) +
    ggplot2::labs(title = panel_title, x = "Threshold probability", y = "Net benefit per 1,000") +
    dca_theme
}
# Build one independent one-row legend from stable keys and obtain every displayed label from dca_labels.
dca_legend_data <- data.frame(x = 0, xend = 1, y = seq_along(dca_legend_keys), yend = seq_along(dca_legend_keys), legend_key = factor(dca_legend_keys, levels = dca_legend_keys))
dca_legend_plot <- ggplot2::ggplot(dca_legend_data) +
  ggplot2::geom_segment(ggplot2::aes(x = x, xend = xend, y = y, yend = yend, color = legend_key, linetype = legend_key), linewidth = 0.75) +
  ggplot2::scale_color_manual(values = dca_legend_colors, limits = dca_legend_keys, breaks = dca_legend_keys, labels = unname(dca_labels[dca_legend_keys]), drop = FALSE, name = NULL) +
  ggplot2::scale_linetype_manual(values = dca_legend_linetypes, limits = dca_legend_keys, breaks = dca_legend_keys, labels = unname(dca_labels[dca_legend_keys]), drop = FALSE, name = NULL) +
  ggplot2::guides(color = ggplot2::guide_legend(nrow = 1, byrow = TRUE, order = 1), linetype = ggplot2::guide_legend(nrow = 1, byrow = TRUE, order = 1)) +
  ggplot2::theme_void(base_family = "sans") +
  ggplot2::theme(plot.background = ggplot2::element_rect(fill = "white", color = NA), 
  legend.background = ggplot2::element_rect(fill = "white", color = NA), 
  legend.box.background = ggplot2::element_rect(fill = "white", color = NA), 
  legend.position = "bottom", legend.direction = "horizontal", 
  legend.text = ggplot2::element_text(size = 8, color = "black", lineheight = 0.88), 
  legend.key = ggplot2::element_rect(fill = "white", color = NA), 
  legend.key.width = grid::unit(15, "pt"), 
  legend.key.height = grid::unit(18, "pt"), 
  legend.spacing.x = grid::unit(0.4, "pt"), 
  legend.margin = ggplot2::margin(0, 0, 0, 0))
dca_legend_grob <- cowplot::get_legend(dca_legend_plot)
# Stack both outcomes and place the single controlled legend underneath.
dca_panel_readmission <- .make_dca_panel("readmission", "Readmission")
dca_panel_mortality <- .make_dca_panel("death", "Mortality")
dca_figure_body <- (dca_panel_readmission / dca_panel_mortality) +
  patchwork::plot_layout(heights = c(1, 1)) +
  patchwork::plot_annotation(tag_levels = "A", theme = ggplot2::theme(plot.tag = ggplot2::element_text(family = "sans", face = "bold", size = 10)))
dca_figure <- cowplot::plot_grid(dca_figure_body, dca_legend_grob, ncol = 1, rel_heights = c(1, 0.10)) +
  ggplot2::theme(plot.background = ggplot2::element_rect(fill = "white", color = NA))
# Display the final publication figure in the notebook.
print(dca_figure)
# Export a vector master, a 600-dpi TIFF, and the plot object for exact regeneration.
fig_dir <- if (exists("project_root", inherits = TRUE)) file.path(get("project_root", inherits = TRUE), "cons", "_figs") else "G:/My Drive/Alvacast/SISTRAT 2023/cons/_figs"
data_out_final <- if (exists("data_out", inherits = TRUE)) get("data_out", inherits = TRUE) else "G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out"
dir.create(fig_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(data_out_final, recursive = TRUE, showWarnings = FALSE)
dca_pdf <- file.path(fig_dir, "Figure19_DCA_patient_bootstrap.pdf")
dca_tiff <- file.path(fig_dir, "Figure19_DCA_patient_bootstrap_600dpi.tiff")
dca_rds <- file.path(data_out_final, "Figure19_DCA_patient_bootstrap_plot.rds")
ggplot2::ggsave(dca_pdf, dca_figure, device = grDevices::cairo_pdf, width = 7.2, height = 7.6, units = "in", bg = "white")
if (requireNamespace("ragg", quietly = TRUE)) ggplot2::ggsave(dca_tiff, dca_figure, device = ragg::agg_tiff, width = 7.2, height = 7.6, units = "in", dpi = 600, compression = "lzw", background = "white") else ggplot2::ggsave(dca_tiff, dca_figure, device = "tiff", width = 7.2, height = 7.6, units = "in", dpi = 600, compression = "lzw", bg = "white")
saveRDS(dca_figure, dca_rds)
message("Saved publication DCA PDF: ", dca_pdf)
message("Saved publication DCA TIFF: ", dca_tiff)
message("Saved publication DCA plot RDS: ", dca_rds)
message("Figure note: Net benefit is expressed as net true cases per 1,000 persons. Mortality ribbons show model-specific pointwise 95% patient-bootstrap confidence intervals. Because the readmission intervals were nearly coincident, the gray ribbon spans the lower and upper limits across all three readmission specifications. The dot-dashed and horizontal reference lines represent treat-all and treat-none strategies, respectively. Only the nonnegative segment of the treat-all strategy is displayed. Readmission CIFs account for death as a competing event, whereas net risk 1-S(t) is included as an audit and should not be interpreted as a competing-risk absolute probability.")

In [ ]:
#| label: holdout-dca-publication-figure-es
#| fig-width: 7.2
#| fig-height: 7.6
#| fig-cap: "Análisis de curvas de decisión en la muestra de validación. Las curvas muestran el beneficio neto, expresado como casos verdaderos netos por 1.000 personas. Las bandas de mortalidad corresponden a IC 95% bootstrap puntuales específicos de cada modelo. En los paneles de Readmisi\u00f3n, la banda gris abarca los IC 95% bootstrap puntuales de las especificaciones, que fueron prácticamente coincidentes. La referencia de punto y raya corresponde a tratar a todas las personas y la referencia horizontal en cero a no tratar a ninguna. Solo se muestra el segmento no negativo de la estrategia de tratar a todas las personas. Las funciones de incidencia acumulada de Readmisi\u00f3n consideran la muerte como evento competidor. La curva de riesgo neto de Readmisi\u00f3n se presenta como análisis complementario porque sus umbrales no representan probabilidades absolutas bajo riesgos competitivos. Los rangos de los ejes varían según el horizonte."
#| warning: false
#| message: false
# Require the packages used to construct and export the Spanish publication figure.
stopifnot(requireNamespace("ggplot2", quietly = TRUE), requireNamespace("dplyr", quietly = TRUE), requireNamespace("patchwork", quietly = TRUE), requireNamespace("cowplot", quietly = TRUE), requireNamespace("scales", quietly = TRUE))
# These five model labels reproduce the wording already used in prediction23_converted_mod.ipynb.
dca_labels_es <- c(death_all = "Mortalidad: todos\nlos predictores", death_shap = "Mortalidad: informado\npor SHAP", read_netrisk = "Readmisi\u00f3n:\nRiesgo neto", read_all = "Readmisi\u00f3n:\nIncidencia acumulada\ntodos los predictores", read_shap = "Readmisi\u00f3n:\nIncidencia acumulada\ninformada por SHAP", treat_all = "Referencia:\nTratar a todas", treat_none = "Referencia:\nNo tratar a ninguna")
# Keep stable internal keys separate from display labels so wording or line-break changes cannot break mappings.
dca_model_keys_es <- c("death_all", "death_shap", "read_netrisk", "read_all", "read_shap")
dca_reference_keys_es <- c("treat_all", "treat_none")
dca_legend_keys_es <- c(dca_model_keys_es, dca_reference_keys_es)
stopifnot(identical(names(dca_labels_es), dca_legend_keys_es))
# Map the original source labels to stable internal keys; no displayed Spanish text is stored here.
dca_model_key_map_es <- c("Mortality Full PH" = "death_all", "Mortality SHAP-informed" = "death_shap", "Readmission net risk 1-S(t)" = "read_netrisk", "Readmission CIF with Full PH mortality" = "read_all", "Readmission CIF with SHAP-informed mortality" = "read_shap")
# Load the dense DCA results only when they are not already available in memory.
if (!exists("dca_boot_all", inherits = TRUE)) {
  dca_rds_candidates_es <- c("G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/pred23_holdout_dca_bootstrap_all_models.rds", "G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out/pred23_holdout_validation_2026_07_17.rds")
  dca_rds_path_es <- dca_rds_candidates_es[file.exists(dca_rds_candidates_es)][1]
  if (is.na(dca_rds_path_es)) stop("No se encontró el archivo RDS con los resultados densos del DCA.")
  dca_loaded_es <- readRDS(dca_rds_path_es)
  if (is.data.frame(dca_loaded_es)) dca_boot_all <- dca_loaded_es else if (is.list(dca_loaded_es) && is.data.frame(dca_loaded_es$results)) dca_boot_all <- dca_loaded_es$results else if (is.list(dca_loaded_es) && is.data.frame(dca_loaded_es$dca_boot_all)) dca_boot_all <- dca_loaded_es$dca_boot_all else stop("El archivo RDS seleccionado no contiene una tabla reconocible de resultados densos del DCA.")
}
# Verify the variables used by the curves, confidence intervals, and reference strategies.
dca_required_es <- c("model_id", "model_label", "outcome", "horizon", "threshold", "in_focus", "event_risk", "net_cases_per1000", "net_cases_per1000_lo", "net_cases_per1000_hi", "B_valid")
stopifnot(all(dca_required_es %in% names(dca_boot_all)), all(dca_boot_all$B_valid == 1000L))
# Reproduce the descriptive windows and create Spanish horizon labels.
dca_plot_data_es <- dca_boot_all |>
  dplyr::filter(in_focus) |>
  dplyr::mutate(model_key = unname(dca_model_key_map_es[model_label]), horizon_label = factor(paste0(horizon, " meses"), levels = paste0(c(6, 12, 36, 60), " meses")))
if (anyNA(dca_plot_data_es$model_key)) stop("Al menos una etiqueta original no está registrada en dca_model_key_map_es.")
dca_plot_data_es$model_key <- factor(dca_plot_data_es$model_key, levels = dca_model_keys_es)
# Define aesthetics using stable keys; these values remain unchanged when displayed labels are edited.
dca_colors_es <- c(death_all = "#24455E", death_shap = "#8A5A2B", read_netrisk = "#8A8A8A", read_all = "#202020", read_shap = "#626262")
dca_fill_colors_es <- c(death_all = "#9FB9CA", death_shap = "#D5B287", read_netrisk = "#D9D9D9", read_all = "#D9D9D9", read_shap = "#D9D9D9")
dca_linetypes_es <- c(death_all = "solid", death_shap = "longdash", read_netrisk = "dotted", read_all = "solid", read_shap = "longdash")
dca_legend_colors_es <- c(dca_colors_es, treat_all = "#333333", treat_none = "#777777")
dca_legend_linetypes_es <- c(dca_linetypes_es, treat_all = "dotdash", treat_none = "solid")
stopifnot(identical(names(dca_legend_colors_es), dca_legend_keys_es), identical(names(dca_legend_linetypes_es), dca_legend_keys_es))
# Create one envelope spanning the nearly coincident readmission confidence intervals.
dca_readmission_envelope_es <- dca_plot_data_es |>
  dplyr::filter(outcome == "readmission") |>
  dplyr::group_by(horizon_label, threshold) |>
  dplyr::summarise(lower = min(net_cases_per1000_lo), upper = max(net_cases_per1000_hi), .groups = "drop")
# Construct smooth treat-all curves from the observed event risk in each displayed threshold window.
dca_reference_specs_es <- dca_plot_data_es |>
  dplyr::group_by(outcome, horizon_label) |>
  dplyr::summarise(event_risk = dplyr::first(event_risk), threshold_min = min(threshold), threshold_max = max(threshold), .groups = "drop")
dca_treat_all_es <- do.call(rbind, lapply(seq_len(nrow(dca_reference_specs_es)), function(i) {
  spec <- dca_reference_specs_es[i, , drop = FALSE]
  threshold_grid <- seq(spec$threshold_min, spec$threshold_max, length.out = 201L)
  data.frame(outcome = spec$outcome, horizon_label = spec$horizon_label, threshold = threshold_grid, net_cases_per1000 = 1000 * (spec$event_risk - threshold_grid) / (1 - threshold_grid))
})) |>
  dplyr::filter(net_cases_per1000 >= 0)
# Format percentages with a decimal comma when half-percentage-point increments are required.
.dca_percent_axis_es <- function(x) {
  pct <- 100 * x
  labels <- ifelse(abs(pct - round(pct)) < 1e-8, paste0(sprintf("%.0f", pct), "%"), paste0(sprintf("%.1f", pct), "%"))
  chartr(".", ",", labels)
}
# Apply a restrained journal-style theme while suppressing panel-specific legends.
dca_theme_es <- ggplot2::theme_classic(base_size = 8.5, base_family = "sans") +
  ggplot2::theme(plot.title = ggplot2::element_text(size = 9.5, face = "bold", hjust = 0), strip.background = ggplot2::element_blank(), strip.text = ggplot2::element_text(size = 8.2, face = "bold"), axis.title = ggplot2::element_text(size = 8.5), axis.text = ggplot2::element_text(size = 7.4, color = "black"), axis.line = ggplot2::element_line(linewidth = 0.35, color = "black"), axis.ticks = ggplot2::element_line(linewidth = 0.35, color = "black"), panel.spacing = grid::unit(8, "pt"), legend.position = "none", plot.margin = ggplot2::margin(4, 5, 2, 5))
# Generate one Spanish outcome panel with model curves, bootstrap uncertainty, and reference strategies.
.make_dca_panel_es <- function(outcome_value, panel_title) {
  panel_data <- dplyr::filter(dca_plot_data_es, outcome == outcome_value)
  panel_ref <- dplyr::filter(dca_treat_all_es, outcome == outcome_value) |>
    dplyr::mutate(reference_key = factor("treat_all", levels = dca_legend_keys_es))
  panel_none <- data.frame(yintercept = 0, reference_key = factor("treat_none", levels = dca_legend_keys_es))
  p <- ggplot2::ggplot(panel_data, ggplot2::aes(x = threshold, y = net_cases_per1000, color = model_key, linetype = model_key, group = model_key)) +
    ggplot2::geom_hline(data = panel_none, ggplot2::aes(yintercept = yintercept, color = reference_key, linetype = reference_key), inherit.aes = FALSE, linewidth = 0.35) +
    ggplot2::geom_line(data = panel_ref, ggplot2::aes(x = threshold, y = net_cases_per1000, color = reference_key, linetype = reference_key, group = 1), inherit.aes = FALSE, linewidth = 0.4)
  if (outcome_value == "readmission") {
    p <- p +
      ggplot2::geom_ribbon(data = dca_readmission_envelope_es, ggplot2::aes(x = threshold, ymin = lower, ymax = upper, group = 1), inherit.aes = FALSE, fill = "#CFCFCF", color = "#A8A8A8", linewidth = 0.20, alpha = 0.55)
  } else {
    p <- p +
      ggplot2::geom_ribbon(ggplot2::aes(ymin = net_cases_per1000_lo, ymax = net_cases_per1000_hi, fill = model_key, group = model_key), color = NA, alpha = 0.30, show.legend = FALSE) +
      ggplot2::scale_fill_manual(values = dca_fill_colors_es, limits = dca_model_keys_es, breaks = dca_model_keys_es, labels = unname(dca_labels_es[dca_model_keys_es]), drop = FALSE, name = NULL)
  }
  p +
    ggplot2::geom_line(linewidth = 0.75) +
    ggplot2::facet_wrap(~horizon_label, nrow = 1, scales = "free") +
    ggplot2::scale_color_manual(values = dca_legend_colors_es, limits = dca_legend_keys_es, breaks = dca_legend_keys_es, labels = unname(dca_labels_es[dca_legend_keys_es]), drop = FALSE, name = NULL) +
    ggplot2::scale_linetype_manual(values = dca_legend_linetypes_es, limits = dca_legend_keys_es, breaks = dca_legend_keys_es, labels = unname(dca_labels_es[dca_legend_keys_es]), drop = FALSE, name = NULL) +
    ggplot2::scale_x_continuous(labels = .dca_percent_axis_es, breaks = scales::breaks_pretty(n = 4), expand = ggplot2::expansion(mult = c(0.01, 0.03))) +
    ggplot2::scale_y_continuous(labels = scales::label_number(accuracy = 1, big.mark = ".", decimal.mark = ","), breaks = scales::breaks_pretty(n = 5), expand = ggplot2::expansion(mult = c(0.04, 0.08))) +
    ggplot2::labs(title = panel_title, x = "Umbral de probabilidad", y = "Beneficio neto por 1.000") +
    dca_theme_es
}
# Build an independent one-row Spanish legend from the stable internal keys.
dca_legend_data_es <- data.frame(x = 0, xend = 1, y = seq_along(dca_legend_keys_es), yend = seq_along(dca_legend_keys_es), legend_key = factor(dca_legend_keys_es, levels = dca_legend_keys_es))
dca_legend_plot_es <- ggplot2::ggplot(dca_legend_data_es) +
  ggplot2::geom_segment(ggplot2::aes(x = x, xend = xend, y = y, yend = yend, color = legend_key, linetype = legend_key), linewidth = 0.75) +
  ggplot2::scale_color_manual(values = dca_legend_colors_es, limits = dca_legend_keys_es, breaks = dca_legend_keys_es, labels = unname(dca_labels_es[dca_legend_keys_es]), drop = FALSE, name = NULL) +
  ggplot2::scale_linetype_manual(values = dca_legend_linetypes_es, limits = dca_legend_keys_es, breaks = dca_legend_keys_es, labels = unname(dca_labels_es[dca_legend_keys_es]), drop = FALSE, name = NULL) +
  ggplot2::guides(color = ggplot2::guide_legend(nrow = 1, byrow = TRUE, order = 1), linetype = ggplot2::guide_legend(nrow = 1, byrow = TRUE, order = 1)) +
  ggplot2::theme_void(base_family = "sans") +
  ggplot2::theme(plot.background = ggplot2::element_rect(fill = "white", color = NA), legend.background = ggplot2::element_rect(fill = "white", color = NA), legend.box.background = ggplot2::element_rect(fill = "white", color = NA), legend.position = "bottom", legend.direction = "horizontal", legend.text = ggplot2::element_text(size = 5.6, color = "black", lineheight = 0.88), legend.key = ggplot2::element_rect(fill = "white", color = NA), legend.key.width = grid::unit(15, "pt"), legend.key.height = grid::unit(22, "pt"), legend.spacing.x = grid::unit(0.3, "pt"), legend.margin = ggplot2::margin(0, 0, 0, 0))
dca_legend_grob_es <- cowplot::get_legend(dca_legend_plot_es)
# Stack both outcomes and place the single controlled Spanish legend underneath.
dca_panel_readmission_es <- .make_dca_panel_es("readmission", "Readmisi\u00f3n")
dca_panel_mortality_es <- .make_dca_panel_es("death", "Mortalidad")
dca_figure_body_es <- (dca_panel_readmission_es / dca_panel_mortality_es) +
  patchwork::plot_layout(heights = c(1, 1)) +
  patchwork::plot_annotation(tag_levels = "A", theme = ggplot2::theme(plot.tag = ggplot2::element_text(family = "sans", face = "bold", size = 10)))
dca_figure_es <- cowplot::plot_grid(dca_figure_body_es, dca_legend_grob_es, ncol = 1, rel_heights = c(1, 0.12)) +
  ggplot2::theme(plot.background = ggplot2::element_rect(fill = "white", color = NA))
# Display the final Spanish publication figure in the notebook.
print(dca_figure_es)
# Export separate Spanish files without overwriting the English publication figure.
fig_dir_es <- if (exists("project_root", inherits = TRUE)) file.path(get("project_root", inherits = TRUE), "cons", "_figs") else "G:/My Drive/Alvacast/SISTRAT 2023/cons/_figs"
data_out_final_es <- if (exists("data_out", inherits = TRUE)) get("data_out", inherits = TRUE) else "G:/My Drive/Alvacast/SISTRAT 2023/data/20241015_out"
dir.create(fig_dir_es, recursive = TRUE, showWarnings = FALSE)
dir.create(data_out_final_es, recursive = TRUE, showWarnings = FALSE)
dca_pdf_es <- file.path(fig_dir_es, "Figura19_DCA_bootstrap_pacientes_es.pdf")
dca_tiff_es <- file.path(fig_dir_es, "Figura19_DCA_bootstrap_pacientes_es_600dpi.tiff")
dca_rds_es <- file.path(data_out_final_es, "Figura19_DCA_bootstrap_pacientes_es_plot.rds")
ggplot2::ggsave(dca_pdf_es, dca_figure_es, device = grDevices::cairo_pdf, width = 7.2, height = 7.6, units = "in", bg = "white")
if (requireNamespace("ragg", quietly = TRUE)) ggplot2::ggsave(dca_tiff_es, dca_figure_es, device = ragg::agg_tiff, width = 7.2, height = 7.6, units = "in", dpi = 600, compression = "lzw", background = "white") else ggplot2::ggsave(dca_tiff_es, dca_figure_es, device = "tiff", width = 7.2, height = 7.6, units = "in", dpi = 600, compression = "lzw", bg = "white")
saveRDS(dca_figure_es, dca_rds_es)
message("Figura DCA en PDF guardada en: ", dca_pdf_es)
message("Figura DCA en TIFF guardada en: ", dca_tiff_es)
message("Objeto RDS de la figura guardado en: ", dca_rds_es)

In [ ]:
#| label: holdout-dca-bootstrap-per1000-2-display
#| message: false
#| warning: false
#| results: asis
# Display-only rebuild of the patient-bootstrap DCA table. Reads the saved
# focus-window CSV exported by holdout-dca-bootstrap-per1000-2, so the
# patient-level bootstrap is not re-run. Two presentation fixes: mortality
# thresholds keep their decimals (1.5% / 2.0% / 2.5% no longer collapse to
# "2%") and the caption describes the windows instead of calling them
# prespecified.
.t0 <- Sys.time()
if (!exists("project_root")) stop("project_root is missing. Run the notebook setup chunks first.", call. = FALSE)
if (!exists("out_dir")) out_dir <- file.path(project_root, "cons", "_out")
dca_boot_focus <- utils::read.csv(file.path(out_dir, "pred23_holdout_dca_bootstrap_focus_windows.csv"), stringsAsFactors = FALSE)
# Format "point (lo to hi)" at one decimal.
.dca_ci <- function(point, lower, upper) sprintf("%.1f (%.1f to %.1f)", point, lower, upper)
# Threshold labels as percentages. Mortality thresholds keep one decimal
# (0.5%, 1.5%, 2.0%, ...); readmission thresholds stay as whole percentages.
.dca_threshold_label <- function(x, outcome) {
  pct <- 100 * x
  lab <- ifelse(outcome == "death", sprintf("%.1f", pct), sprintf("%.0f", pct))
  paste0(lab, "%")
}
dca_boot_display <- data.frame(
  Model = dca_boot_focus$model_label,
  Horizon = dca_boot_focus$horizon,
  Threshold = .dca_threshold_label(dca_boot_focus$threshold, dca_boot_focus$outcome),
  `Observed events per 1,000 (95% CI)` = .dca_ci(dca_boot_focus$event_risk * 1000, dca_boot_focus$event_risk_lo * 1000, dca_boot_focus$event_risk_hi * 1000),
  `Net true cases per 1,000 vs treat-none (95% CI)` = .dca_ci(dca_boot_focus$net_cases_per1000, dca_boot_focus$net_cases_per1000_lo, dca_boot_focus$net_cases_per1000_hi),
  `Unnecessary interventions avoided per 1,000 vs treat-all (95% CI)` = .dca_ci(dca_boot_focus$avoided_per1000, dca_boot_focus$avoided_per1000_lo, dca_boot_focus$avoided_per1000_hi),
  check.names = FALSE
)
print(
  htmltools::browsable(
    htmltools::tagList(
      htmltools::HTML(
        knitr::kable(
          dca_boot_display,
          format = "html",
          align = c("l", "c", "c", "c", "c", "c"),
          caption = "Patient-bootstrap decision-curve results within descriptive windows based on 0.5 to 2 times the observed event risk."
        )
      )
    )
  )
)
message(sprintf("Display table rebuilt from saved CSV in %.2f minutes.", as.numeric(difftime(Sys.time(), .t0, units = "mins"))))

## Reclassification: NRI & IDI (held-out 20%)

Competing-risk-aware reclassification (readmission via Aalen-Johansen IPCW, mortality via standard IPCW), mirroring `prediction225`. `old` = SHAP implemented death (`best_perf2`), `new` = Full PH primary death (`best_perf1`): positive NRI/IDI = the Full PH primary reclassifies better than the parsimonious implementation. Readmission is shared by both models, so its NRI/IDI is 0 by construction — the **mortality** rows carry the signal. NRI/IDI is a secondary metric; its intervals are split-to-split percentiles, not CIs.

In [ ]:
#| label: holdout-nri-idi-run
#| message: false

source("cons/_alt_scripts/nri_idi_from_results_boot.R")          # AJ competing-aware
source("cons/_hist_scripts/make_nri_idi_epidemiology_table.R")

NRI_IDI_HORIZONS   <- c(6, 12, 36, 60)
NRI_IDI_CUT_POINTS <- c(0.05, 0.10, 0.20)

nri_idi_holdout <- run_nri_idi_from_results_boot(
  results_boot_old = results_boot_val_bp2,   # SHAP implemented death
  results_boot_new = results_boot_val_bp1,   # Full PH primary death
  horizons       = NRI_IDI_HORIZONS,
  cut_points     = NRI_IDI_CUT_POINTS,
  old_label      = "SHAP implemented (best_perf2)",
  new_label      = "Full PH primary (best_perf1)",
  readmit_method = "aalen-johansen",
  output_dir     = file.path(out_dir, "nri_idi_holdout"),
  prefix         = "nri_idi_holdout",
  save_raw       = FALSE
)
cat(sprintf("Readmission CR=%d | IPCW fallback=%d\n",
            nri_idi_holdout$config$n_readmit_competing_risk,
            nri_idi_holdout$config$n_readmit_ipcw_fallback))

nri_idi_tab <- make_nri_idi_epidemiology_table(
  reclass_object = nri_idi_holdout,
  output_dir = file.path(out_dir, "nri_idi_epi_holdout"),
  prefix     = "nri_idi_epi_holdout",
  old_label  = "SHAP implemented (best_perf2)",
  new_label  = "Full PH primary (best_perf1)"
)

source(file.path(project_root, 
  "cons",
  "_alt_scripts",
  "relabel_nri_idi_table.R"
  )
)
nri_idi_fixed <- relabel_nri_idi_table(
  nri_idi_tab$table,
  old_label = "SHAP implemented (best_perf2)",   # referencia
  new_label = "Full PH primary (best_perf1)"     # actualizado
)
nri_idi_model_table <- nri_idi_fixed$model_table

utils::write.csv(nri_idi_model_table,
                 file.path(out_dir, "table_nri_holdout.csv"), row.names = FALSE)

In [ ]:
#| label: holdout-nri-idi-table
nri_idi_fixed$table |>
  knitr::kable("markdown", caption = nri_idi_fixed$caption)

## ipeval bootstrap consistency check (held-out 20%)

`ipeval::ip_score` with the dummy-treatment IPW trick (random 50/50 → weight 2). Strata-correct `predictRisk` vectors are fed directly (ipeval's internal `predict_cox` mishandles stratified Cox). Cross-check: ipeval `auc` ↔ Uno's C-index, `oeratio` ↔ E:O, `brier` ↔ point-horizon Brier. Requires `renv::install("ipeval")`.

In [ ]:
#| label: holdout-resource-engines
if (!exists("project_root")) project_root <- gsub("/cons$", "", here::here())
source(file.path(project_root, "cons/_alt_scripts/validate_holdout_metrics.R"))  # -> ici_bootstrap_holdout()
source(file.path(project_root, "cons/_alt_scripts/validate_holdout_ipeval.R"))   # -> ipeval_holdout(parallel=)
"parallel" %in% names(formals(ipeval_holdout))   # ipeval_holdout now takes parallel
exists("ici_bootstrap_holdout")                  # the ICI bootstrap is loaded


In [ ]:
#| label: holdout-ipeval-run
#| message: false
library(future); library(future.apply)
if (!inherits(future::plan(), "multisession")) future::plan(future::multisession, workers = 20)
options(future.globals.maxSize = 10 * 1024^3)

B_BOOT <- 1000L
ipeval_res <- if (requireNamespace("ipeval", quietly = TRUE)) {
  ipeval_holdout(models, train_list, val_list, horizons = DCA_HORIZONS,
                 bootstrap = B_BOOT, seed = 2125L, parallel = TRUE, verbose = TRUE)
} else { message("ipeval not installed; run renv::install('ipeval') to enable."); NULL }


~ 13 minutes

In [ ]:
#| label: holdout-ipeval-table
if (!is.null(ipeval_res)) {
  knitr::kable(ipeval_res$pooled, "markdown", digits = 3,
    caption = "ipeval (held-out 20%, dummy-treatment IPW): AUC / Brier / O:E with bootstrap 95% CI")
}

In [ ]:
#| label: holdout-ipeval-plot
#| fig-width: 11
#| fig-height: 4.5
tnr <- "Times New Roman"

plot_ipeval_epi <- function(ipeval_res, tnr = "Times New Roman") {
  p <- ipeval_res$pooled
  mk <- function(m, lab) data.frame(
    risk = p$risk, model = p$model, horizon = p$horizon, metric = lab,
    value = p[[paste0(m, "_mean")]],
    lower = p[[paste0(m, "_lower_mean")]], upper = p[[paste0(m, "_upper_mean")]],
    stringsAsFactors = FALSE)
  long <- rbind(mk("auc", "AUC(t)"), mk("brier", "Brier(t)"))
  lab_map <- c("readmit::best_perf1" = "Readmission (shared)",
               "death::best_perf1"   = "Mortality best_perf1 (Full PH)",
               "death::best_perf2"   = "Mortality best_perf2 (SHAP)")
  long$model_lab <- factor(unname(lab_map[long$model]), levels = unname(lab_map))
  long$metric    <- factor(long$metric, levels = c("AUC(t)", "Brier(t)"))
  long <- long[order(long$model_lab, long$horizon), ]   # ensure lines connect correctly
  pd <- position_dodge(width = 2.5)
  href <- data.frame(metric = factor("AUC(t)", levels = levels(long$metric)),
                     y = 0.5)                          # chance line
  cols <- c("Readmission (shared)"           = "#2166AC",
            "Mortality best_perf1 (Full PH)" = "#B2182B",
            "Mortality best_perf2 (SHAP)"    = "#E08214") #best_perf2
  ggplot2::ggplot(long, ggplot2::aes(horizon, value, color = model_lab, fill = model_lab, group = model_lab)) +
    ggplot2::geom_hline(data = href, ggplot2::aes(yintercept = y),
                        linetype = "dashed", color = "grey50", linewidth = 0.4) +
    ggplot2::geom_ribbon(ggplot2::aes(ymin = lower, ymax = upper), alpha = 0.15, color = NA, position = pd) +
    ggplot2::geom_line(linewidth = 0.8, position = pd) +
    ggplot2::geom_point(size = 1.8, position = pd) +
    ggplot2::geom_errorbar(ggplot2::aes(ymin = lower, ymax = upper),
                           width = 1.2, linewidth = 0.4, alpha = 0.7, position = pd) +
    ggplot2::facet_wrap(~metric, scales = "free_y", nrow = 1) +
    ggplot2::scale_color_manual(values = cols) + ggplot2::scale_fill_manual(values = cols) +
    ggplot2::scale_x_continuous(breaks = sort(unique(long$horizon))) +
    ggplot2::labs(x = "Months since discharge", y = "Value (bootstrap 95% CI)",
                  color = NULL, fill = NULL,
                  title = NULL)+#"ipeval (held-out 20%): discrimination (AUC) and accuracy (Brier)") +
    ggplot2::theme_bw(base_size = 13, base_family = tnr) +
    ggplot2::theme(legend.position = "bottom", panel.grid.minor = ggplot2::element_blank(),
                   strip.background = ggplot2::element_rect(fill = "grey95", color = "grey80"),
                   strip.text = ggplot2::element_text(face = "bold"))
}

if (!is.null(ipeval_res)) {
  g_ip <- plot_ipeval_epi(ipeval_res, tnr)
  print(g_ip)
  ggplot2::ggsave(file.path(figs_out, "holdout_ipeval_bootstrap.png"),
                  g_ip, width = 28, height = 11, units = "cm", dpi = 600)
}

In [ ]:
#| label: holdout-cindex-bootstrap
#| message: false
suppressPackageStartupMessages({ library(survival) })
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/evaluate_dual_cox_holdout_dualscore.R"))
stopifnot(exists(".dual_cox_safe_concordance_coherent"),
          exists("results_boot_val_bp1"), exists("results_boot_val_bp2"))

B_CIDX            <- 1000L                 # smoke-test with 50 first
HZ_CIDX           <- c(6, 12, 36, 60)
CIDX_SEED         <- 2125L
USE_PARALLEL_CIDX <- TRUE

# ---- pooled lp AND risk (both scores) from the frozen raw_predictions ----
re_dat  <- .holdout_pool_dualscore(results_boot_val_bp1, val_list, "readmission")  # shared bp1 == bp2
de1_dat <- .holdout_pool_dualscore(results_boot_val_bp1, val_list, "death")
de2_dat <- .holdout_pool_dualscore(results_boot_val_bp2, val_list, "death")

# ---- defensive parallel backend (same as before) ----
if (USE_PARALLEL_CIDX && requireNamespace("future", quietly = TRUE) &&
    !inherits(future::plan(), "multisession")) {
  options(parallelly.makeNodePSOCK.setup_strategy = "sequential")
  options(parallelly.makeNodePSOCK.connectTimeout = 600)
  nw <- min(12L, max(1L, parallelly::availableCores() %/% 2L))
  ok <- tryCatch({ future::plan(future::multisession, workers = nw); TRUE },
                 error = function(e) { message("seq fallback: ", conditionMessage(e)); FALSE })
  if (!ok) { USE_PARALLEL_CIDX <- FALSE; future::plan(future::sequential) }
  message("Uno's C bootstrap workers: ", if (ok) nw else 0L)
}

cidx_re  <- .holdout_cindex_boot_dualscore(re_dat,  HZ_CIDX, B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX)
cidx_de1 <- .holdout_cindex_boot_dualscore(de1_dat, HZ_CIDX, B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX)
cidx_de2 <- .holdout_cindex_boot_dualscore(de2_dat, HZ_CIDX, B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX)

# ---- format as "point (lower, upper)" by ScoreType, same look as the original table ----
.fmt_cidx <- function(cidx_tab, score_type) {
  d <- cidx_tab[cidx_tab$ScoreType == score_type, ]
  setNames(sprintf("%.4f (%.4f, %.4f)", d$C, d$C_lower, d$C_upper), as.character(d$Horizon))
}

cat("== READMISSION (best_perf1 == best_perf2) | Uno's C, bootstrap B=", B_CIDX, " ==\n", sep = "")
cat("  by lp:   "); print(.fmt_cidx(cidx_re, "lp"))
cat("  by risk: "); print(.fmt_cidx(cidx_re, "risk"))
cat("\n== DEATH best_perf1 (Full PH) | Uno's C ==\n")
cat("  by lp:   "); print(.fmt_cidx(cidx_de1, "lp"))
cat("  by risk: "); print(.fmt_cidx(cidx_de1, "risk"))
cat("\n== DEATH best_perf2 (SHAP)   | Uno's C ==\n")
cat("  by lp:   "); print(.fmt_cidx(cidx_de2, "lp"))
cat("  by risk: "); print(.fmt_cidx(cidx_de2, "risk"))
cat("\nUno's C with IPCW (timewt = n/G2); 95% CI = nonparametric bootstrap of the",
    "held-out test patients (model frozen, no refit), MI-pooled predictions.",
    "lp = linear predictor (audit/stability check); risk = 1-S(t) per horizon,",
    "the PRIMARY discrimination score under stratified baseline hazards.\n")

# ============================ IBS + combined C/IBS + null IBS =========================
# IBS does not depend on the ranking score, so it stays a SINGLE arm here (unchanged).
# The readmission-CIF sensitivity split (1-S(t) vs joint CIF) is a SEPARATE, later
# change (point 3), not part of this cell.
source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/ibs_window_bootstrap_holdout.R"))
IBS_READMIT_METHOD <- "ipcw"   # reproduces the reported readmission IBS (death as censoring)

ibs_bp1 <- ibs_window_bootstrap_core(results_boot_val_bp1, B = B_CIDX, seed = CIDX_SEED,
             eval_times = c(3, HZ_CIDX), readmit_method = IBS_READMIT_METHOD, verbose = FALSE)
ibs_bp2 <- ibs_window_bootstrap_core(results_boot_val_bp2, B = B_CIDX, seed = CIDX_SEED,
             eval_times = c(3, HZ_CIDX), readmit_method = IBS_READMIT_METHOD, verbose = FALSE)

.combine_c_ibs_dualscore <- function(cidx_tab, ibs_df, which_risk) {
  ci_lp   <- .fmt_cidx(cidx_tab, "lp")
  ci_risk <- .fmt_cidx(cidx_tab, "risk")
  d   <- ibs_df[ibs_df$risk == which_risk & is.finite(ibs_df$mean), ]
  nul <- attr(ibs_df, "null"); nul <- nul[nul$risk == which_risk & is.finite(nul$mean), ]
  fmt <- function(p, l, h) sprintf("%.4f (%.4f, %.4f)", p, l, h)
  ib  <- setNames(fmt(d$point, d$q025, d$q975),       as.character(d$horizon))
  i0  <- setNames(fmt(nul$point, nul$q025, nul$q975), as.character(nul$horizon))
  sk  <- setNames(sprintf("%.3f", 1 - d$point / nul$point), as.character(d$horizon))
  ib["Global"] <- ib["60"]; i0["Global"] <- i0["60"]; sk["Global"] <- sk["60"]
  ord <- c("6", "12", "36", "60", "Global")
  data.frame(Horizon = ord,
             `Uno's C, lp (95% CI)`   = unname(ci_lp[ord]),
             `Uno's C, risk (95% CI)` = unname(ci_risk[ord]),
             `IBS (95% CI)`      = unname(ib[ord]),
             `IBS null (95% CI)` = unname(i0[ord]),
             `IBS skill`         = unname(sk[ord]),
             check.names = FALSE, stringsAsFactors = FALSE)
}
tab_re  <- .combine_c_ibs_dualscore(cidx_re,  ibs_bp1, "readmission")
tab_de1 <- .combine_c_ibs_dualscore(cidx_de1, ibs_bp1, "death")
tab_de2 <- .combine_c_ibs_dualscore(cidx_de2, ibs_bp2, "death")

cat("\n== Combined C (lp, risk) + IBS | READMISSION (best_perf1 == best_perf2) ==\n"); print(tab_re,  row.names = FALSE)
cat("\n== Combined C (lp, risk) + IBS | DEATH best_perf1 (Full PH) ==\n");            print(tab_de1, row.names = FALSE)
cat("\n== Combined C (lp, risk) + IBS | DEATH best_perf2 (SHAP) ==\n");               print(tab_de2, row.names = FALSE)

.null_msg <- function(tab, lbl)
  paste0(lbl, " | ", paste(sprintf("%s: IBS0 %s, skill %s", tab$Horizon,
         tab$`IBS null (95% CI)`, tab$`IBS skill`), collapse = " | "))
message("Null-model IBS (intercept-only; same bootstrap and estimand) and skill = 1 - IBS/IBS0:")
message("  ", .null_msg(tab_re,  "Readmission(shared)"))
message("  ", .null_msg(tab_de1, "Death best_perf1"))
message("  ", .null_msg(tab_de2, "Death best_perf2"))

# tidy IBS table for saving (one row per risk/model/horizon, point + 95% bootstrap CI)
ibs_window <- rbind(
  cbind(model = "shared",     ibs_bp1[ibs_bp1$risk == "readmission", ]),
  cbind(model = "best_perf1", ibs_bp1[ibs_bp1$risk == "death", ]),
  cbind(model = "best_perf2", ibs_bp2[ibs_bp2$risk == "death", ]))
rownames(ibs_window) <- NULL

# tidy raw C table (both scores, all horizons) for saving alongside the combined csv
cidx_window <- rbind(
  cbind(model = "readmit_shared",   cidx_re),
  cbind(model = "death_best_perf1", cidx_de1),
  cbind(model = "death_best_perf2", cidx_de2))

if (exists("out_dir")) {
  utils::write.csv(rbind(cbind(model = "readmit_shared",   tab_re),
                         cbind(model = "death_best_perf1", tab_de1),
                         cbind(model = "death_best_perf2", tab_de2)),
                   file.path(out_dir, "holdout_c_ibs_combined.csv"), row.names = FALSE)
  utils::write.csv(cidx_window,
                   file.path(out_dir, "holdout_cindex_dualscore_raw.csv"), row.names = FALSE)
}

# Comparison between death models 
delta_c_death_models <- .holdout_delta_c_dualscore_ij(
  de1_dat, de2_dat, horizons = HZ_CIDX, label_A = "best_perf1", label_B = "best_perf2")

utils::write.csv(delta_c_death_models, file.path(out_dir, "holdout_delta_c_death_models_dualscore.csv"), row.names = FALSE)

In [ ]:
#| label: holdout-cindex-within-strata
#| message: false

source(file.path(if (exists("project_root")) project_root else getwd(),
                 "cons/_alt_scripts/evaluate_dual_cox_holdout_dualscore.R"))
stopifnot(exists("re_dat"), exists("de1_dat"), exists("de2_dat"),
          exists(".holdout_pool_strata_for"), exists(".holdout_cindex_boot_within_strata"))

strata_re  <- .holdout_pool_strata_for(results_boot_val_bp1, val_list, models$best_perf1$readmit, "readmission")
strata_de1 <- .holdout_pool_strata_for(results_boot_val_bp1, val_list, models$best_perf1$death,   "death")
strata_de2 <- .holdout_pool_strata_for(results_boot_val_bp2, val_list, models$best_perf2$death,    "death")

cidx_re_ws  <- rbind(
  .holdout_cindex_boot_within_strata(re_dat,  strata_re,  HZ_CIDX, score_type = "lp",   B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX),
  .holdout_cindex_boot_within_strata(re_dat,  strata_re,  HZ_CIDX, score_type = "risk", B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX)
)
cidx_de1_ws <- rbind(
  .holdout_cindex_boot_within_strata(de1_dat, strata_de1, HZ_CIDX, score_type = "lp",   B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX),
  .holdout_cindex_boot_within_strata(de1_dat, strata_de1, HZ_CIDX, score_type = "risk", B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX)
)
cidx_de2_ws <- rbind(
  .holdout_cindex_boot_within_strata(de2_dat, strata_de2, HZ_CIDX, score_type = "lp",   B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX),
  .holdout_cindex_boot_within_strata(de2_dat, strata_de2, HZ_CIDX, score_type = "risk", B = B_CIDX, seed = CIDX_SEED, parallel = USE_PARALLEL_CIDX)
)

cat("== READMISSION (best_perf1 == best_perf2) | Uno's C within stratum, bootstrap B=", B_CIDX, " ==\n", sep = "")
cat("  by lp:   "); print(.fmt_cidx(cidx_re_ws, "lp"))
cat("  by risk: "); print(.fmt_cidx(cidx_re_ws, "risk"))
cat("\n== DEATH best_perf1 (Full PH) | Uno's C C within stratum ==\n")
cat("  by lp:   "); print(.fmt_cidx(cidx_de1_ws, "lp"))
cat("  by risk: "); print(.fmt_cidx(cidx_de1_ws, "risk"))
cat("\n== DEATH best_perf2 (SHAP)   | Uno's C C within stratum ==\n")
cat("  by lp:   "); print(.fmt_cidx(cidx_de2_ws, "lp"))
cat("  by risk: "); print(.fmt_cidx(cidx_de2_ws, "risk"))

# comparación global vs intraestrato (score risk, la primaria)
cindex_within_strata <- rbind(
  cbind(model = "readmit_shared",   cidx_global = "risk", cidx_re_ws),
  cbind(model = "death_best_perf1", cidx_global = "risk", cidx_de1_ws),
  cbind(model = "death_best_perf2", cidx_global = "risk", cidx_de2_ws)
)
knitr::kable(
  cindex_within_strata[, c("model", "Horizon", "ScoreType", "C", "C_lower", "C_upper", "B_valid")],
  "markdown", digits = 4,
  caption = "Held-out 20%: Uno's C-index within strata (restricted to pairs in the same strata)") |> print()

cat("\nGlobal (pooled, all-pairs) vs intraestrato, score = risk:\n")
cat(sprintf("  Readmission:      global %.4f  |  intraestrato %.4f\n",
    cidx_re$C[cidx_re$Horizon == "Global" & cidx_re$ScoreType == "risk"],
    cidx_re_ws$C[cidx_re_ws$Horizon == "Global" & cidx_re_ws$ScoreType == "risk"]))
cat(sprintf("  Death best_perf1: global %.4f  |  intraestrato %.4f\n",
    cidx_de1$C[cidx_de1$Horizon == "Global" & cidx_de1$ScoreType == "risk"],
    cidx_de1_ws$C[cidx_de1_ws$Horizon == "Global" & cidx_de1_ws$ScoreType == "risk"]))
cat(sprintf("  Death best_perf2: global %.4f  |  intraestrato %.4f\n",
    cidx_de2$C[cidx_de2$Horizon == "Global" & cidx_de2$ScoreType == "risk"],
    cidx_de2_ws$C[cidx_de2_ws$Horizon == "Global" & cidx_de2_ws$ScoreType == "risk"]))

if (exists("out_dir")) {
  utils::write.csv(cindex_within_strata,
                   file.path(out_dir, "holdout_cindex_within_strata.csv"), row.names = FALSE)
}


In [ ]:
#| label: holdout-ici-bootstrap-run
#| message: false
library(future); library(future.apply)
if (!inherits(future::plan(), "multisession")) future::plan(future::multisession, workers = ms())
options(future.globals.maxSize = 10 * 1024^3)

ICI_B <- 1000L
ici_boot_readmit <- rbind(
  .holdout_bootstrap_calibration_readmit_from_raw(results_boot_val_bp1,     times = c(6,12,36,60), B = ICI_B, seed = 2125L, model_label = "readmit::netrisk", parallel = TRUE, verbose = TRUE),
  .holdout_bootstrap_calibration_readmit_from_raw(results_boot_val_bp1_cif, times = c(6,12,36,60), B = ICI_B, seed = 2125L, model_label = "readmit::bp1_cif",  parallel = TRUE, verbose = TRUE),
  .holdout_bootstrap_calibration_readmit_from_raw(results_boot_val_bp2_cif, times = c(6,12,36,60), B = ICI_B, seed = 2125L, model_label = "readmit::bp2_cif",  parallel = TRUE, verbose = TRUE)
)
ici_boot_death <- ici_bootstrap_holdout(models, train_list, val_list, times = c(6,12,36,60), B = ICI_B, seed = 2125L, parallel = TRUE, verbose = TRUE)
ici_boot <- rbind(ici_boot_readmit, ici_boot_death)
utils::write.csv(ici_boot, file.path(out_dir, "pred23_holdout_ici_bootstrap.csv"), row.names = FALSE)


~ 88 minutes

In [ ]:
#| label: holdout-ici-bootstrap-table
#| message: false
#| results: asis
library(htmltools)

ici_epi_table <- function(ici_boot, digits = 4) {
  d <- as.data.frame(ici_boot)
  f <- function(e, l, h) ifelse(is.na(e), "\u2014",
        sprintf("%.*f (%.*f, %.*f)", digits, e, digits, l, digits, h))
  out <- data.frame(
    Outcome = ifelse(d$risk == "readmission", "Readmission", "Mortality"),
    Model = dplyr::recode(d$model,
      "readmit::shared"   = "SHAP-informed (shared)",
      "death::best_perf1" = "Full PH (best_perf1)",
      "death::best_perf2" = "SHAP 13-var (best_perf2)"),
    Horizon = d$horizon,
    ICI = f(d$ici, d$ici_lo, d$ici_hi),
    ECE = f(d$ece, d$ece_lo, d$ece_hi),
    OE  = ifelse(is.na(d$eo), "\u2014", sprintf("%.2f (%.2f, %.2f)", d$eo, d$eo_lo, d$eo_hi)),
    stringsAsFactors = FALSE, check.names = FALSE)
  out[order(factor(out$Outcome, c("Readmission", "Mortality")), out$Model, out$Horizon), ]
}

# Generic grouped browsable table (reuse for ipeval / threshold tables too)
.browsable_grouped <- function(df, group_col, body_cols, header_labels, caption, footnote,
                               font = "'Times New Roman', serif") {
  th <- sprintf("position:sticky;top:0;background:#f3f3f3;border-bottom:2px solid #ccc;padding:6px 10px;text-align:center;white-space:nowrap;font-weight:bold;")
  td <- "border-bottom:1px solid #eee;padding:4px 10px;white-space:nowrap;"
  gh <- "background:#e8eef5;font-weight:bold;padding:5px 10px;border-bottom:1px solid #ccc;"
  header <- htmltools::tags$thead(htmltools::tags$tr(
    lapply(header_labels, function(nm) htmltools::tags$th(style = th, nm))))
  groups <- split(df, factor(df[[group_col]], levels = unique(df[[group_col]])))
  rows <- list()
  for (g in names(groups)) {
    rows[[length(rows) + 1L]] <- htmltools::tags$tr(
      htmltools::tags$td(colspan = length(body_cols), style = gh, g))
    gd <- groups[[g]]
    for (i in seq_len(nrow(gd)))
      rows[[length(rows) + 1L]] <- htmltools::tags$tr(lapply(seq_along(body_cols), function(j)
        htmltools::tags$td(style = paste0(td, "text-align:", if (j == 1) "left" else "center", ";"),
                           as.character(gd[i, body_cols[j]]))))
  }
  htmltools::browsable(htmltools::tags$div(
    htmltools::tags$div(style = paste0("font-weight:bold;margin-bottom:6px;font-family:", font, ";"), caption),
    htmltools::tags$div(style = paste0("max-height:550px;overflow:auto;border:1px solid #ddd;font-family:", font, ";font-size:13px;"),
      htmltools::tags$table(style = "border-collapse:collapse;width:max-content;min-width:100%;",
        header, htmltools::tags$tbody(rows))),
    htmltools::tags$div(style = paste0("font-size:11px;color:#555;margin-top:6px;max-width:780px;font-family:", font, ";"), footnote)))
}

if (exists("ici_boot")) {
  tab <- ici_epi_table(ici_boot)
  tab$Model[duplicated(paste(tab$Outcome, tab$Model))] <- ""
  .browsable_grouped(
    tab, group_col = "Outcome",
    body_cols     = c("Model", "Horizon", "ICI", "ECE", "OE"),
    header_labels = c("Model", "Horizon, mo", "ICI (95% CI)", "ECE (95% CI)", "E:O (95% CI)"),
    caption  = "Held-out (20%) calibration indices with bootstrap 95% CI.",
    footnote = paste0("ICI, integrated calibration index; ECE, estimated calibration error; ",
      "E:O, mean predicted / observed risk. Readmission observed risk via Aalen-Johansen ",
      "(death competing); mortality via Kaplan-Meier. Lower ICI/ECE = better; E:O 1.0 = perfect."))
}


In [ ]:
#| label: holdout-ici-bootstrap-table2
ici_boot |>
  dplyr::transmute(model, horizon,
    ICI  = sprintf("%.4f (%.4f-%.4f)", ici, ici_lo, ici_hi),
    ECE  = sprintf("%.4f (%.4f-%.4f)", ece, ece_lo, ece_hi),
    `E:O`= sprintf("%.2f (%.2f-%.2f)",  eo,  eo_lo,  eo_hi)) |>
  knitr::kable("markdown",
    caption = "Held-out 20%: calibration indices with bootstrap 95% CI (predictions fixed, validation rows resampled)")


In [ ]:
#| label: holdout-ici-bootstrap-plot
#| fig-width: 8
#| fig-height: 9
library(ggplot2); library(patchwork)
tnr <- "Times New Roman"
plot_ici_boot_epi <- function(ici_boot, tnr = "Times New Roman",
                              metric_titles = c(ici = "Integrated\nCalibration\nIndex (ICI)",
                                                ece = "Estimated\nCalibration\nError (ECE)",
                                                eo  = "Expected:Observed\nratio (E:O)")) {
  d <- as.data.frame(ici_boot)
  mk <- function(m) data.frame(risk = d$risk, model = d$model, horizon = d$horizon, metric = m,
    value = d[[m]], lower = d[[paste0(m, "_lo")]], upper = d[[paste0(m, "_hi")]])
  long <- do.call(rbind, lapply(names(metric_titles), mk))
  long$outcome <- factor(ifelse(long$risk == "readmission", "Readmission", "Mortality"),
                         levels = c("Readmission", "Mortality"))
  long$model_lab <- factor(dplyr::case_when(
    long$model == "readmit::netrisk"  ~ "Readmission:\nNet risk",
    long$model == "readmit::bp1_cif"  ~ "Readmission:\nCum. incidence\nall predictors",
    long$model == "readmit::bp2_cif"  ~ "Readmission:\nCum. incidence\nSHAP-informed",
    long$model == "death::best_perf1" ~ "Mortality:\nAll predictors",
    long$model == "death::best_perf2" ~ "Mortality:\nSHAP-informed",
    TRUE ~ long$model),
    levels = c("Readmission:\nNet risk",
               "Readmission:\nCum. incidence\nall predictors",
               "Readmission:\nCum. incidence\nSHAP-informed",
               "Mortality:\nAll predictors",
               "Mortality:\nSHAP-informed"))
  cols <- c("Readmission:\nNet risk" = "#9ECAE1",
            "Readmission:\nCum. incidence\nall predictors" = "#4292C6",
            "Readmission:\nCum. incidence\nSHAP-informed" = "#08519C",
            "Mortality:\nAll predictors" = "#B2182B",
            "Mortality:\nSHAP-informed" = "#E08214")
  shp  <- c("Readmission:\nNet risk" = 21,
            "Readmission:\nCum. incidence\nall predictors" = 23,
            "Readmission:\nCum. incidence\nSHAP-informed" = 25,
            "Mortality:\nAll predictors" = 24,
            "Mortality:\nSHAP-informed" = 22)
  theme_epi <- theme_classic(base_size = 13, base_family = tnr) +
    theme(legend.position = "bottom", legend.title = element_blank(), legend.key.width = unit(1.2, "lines"),
          strip.background = element_blank(), strip.text = element_text(face = "bold", size = 12),
          axis.title = element_text(face = "bold"), axis.text = element_text(color = "black"),
          panel.grid.major.y = element_line(color = "grey92", linewidth = 0.3),
          panel.spacing = unit(0.8, "lines"), plot.tag = element_text(face = "bold", size = 16, family = tnr))
  mk_panel <- function(m, ytitle, ref = NA_real_, show_x = FALSE, show_strip = FALSE, expand0 = FALSE) {
    dd <- long[long$metric == m, ]
    dd <- dd[order(dd$model_lab, dd$horizon), ]
    pd <- position_dodge(width = 2.5)
    g <- ggplot(dd, aes(horizon, value, color = model_lab, shape = model_lab, group = model_lab))
    if (is.finite(ref)) g <- g + geom_hline(yintercept = ref, linetype = "22", color = "grey45", linewidth = 0.4)
    g <- g + geom_errorbar(aes(ymin = lower, ymax = upper), width = 1.2, linewidth = 0.5, alpha = 0.85, position = pd) +
      geom_line(linewidth = 0.7, alpha = 0.9, position = pd) + geom_point(size = 2.5, fill = "white", stroke = 0.9, position = pd) +
      facet_wrap(~ outcome, nrow = 1, scales = "free_y") +
      scale_color_manual(values = cols, drop = FALSE) + scale_shape_manual(values = shp, drop = FALSE) +
      scale_x_continuous(breaks = c(6, 12, 36, 60)) + labs(x = "Months since discharge", y = ytitle) + theme_epi
    if (expand0) g <- g + expand_limits(y = 0)
    if (!show_strip) g <- g + theme(strip.text = element_blank())
    if (!show_x) g <- g + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())
    g
  }
  pA <- mk_panel("ici", metric_titles["ici"], show_x = FALSE, show_strip = TRUE,  expand0 = TRUE)
  pB <- mk_panel("ece", metric_titles["ece"], show_x = FALSE, show_strip = FALSE, expand0 = TRUE)
  pC <- mk_panel("eo",  metric_titles["eo"],  ref = 1, show_x = TRUE, show_strip = FALSE)
  (pA / pB / pC) + plot_layout(guides = "collect") + plot_annotation(tag_levels = "A") &
    theme(legend.position = "bottom")
}

if (exists("ici_boot")) {
  g_ici <- plot_ici_boot_epi(ici_boot, tnr); print(g_ici)
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot2_panel.tiff"), g_ici,
                  width = 17.8, height = 20, units = "cm", dpi = 600, compression = "lzw")
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot2_panel.png"), g_ici,
                  width = 17.8, height = 20, units = "cm", dpi = 600)
}


In [ ]:
#| label: holdout-ici-bootstrap-plot3
#| fig-width: 8
#| fig-height: 6.5

library(ggplot2); library(patchwork)
tnr <- "Times New Roman"

plot_ici_boot_epi <- function(ici_boot, tnr = "Times New Roman",
                              metric_titles = c(ici = "Integrated\nCalibration Index (ICI)",
                                                ece = "Expected\nCalibration Error (ECE)"),
                              dodge = 3) {
  d <- as.data.frame(ici_boot)
  mk <- function(m) data.frame(risk = d$risk, model = d$model, horizon = d$horizon, metric = m,
    value = d[[m]], lower = d[[paste0(m, "_lo")]], upper = d[[paste0(m, "_hi")]])
  long <- do.call(rbind, lapply(names(metric_titles), mk))
  long$outcome <- factor(ifelse(long$risk == "readmission", "Readmission", "Mortality"),
                         levels = c("Readmission", "Mortality"))
  long$model_lab <- factor(dplyr::case_when(
    long$model == "readmit::netrisk"  ~ "Readmission:\nNet risk",
    long$model == "readmit::bp1_cif"  ~ "Readmission:\nCum. incidence\nall predictors",
    long$model == "readmit::bp2_cif"  ~ "Readmission:\nCum. incidence\nSHAP-informed",
    long$model == "death::best_perf1" ~ "Mortality:\nAll predictors",
    long$model == "death::best_perf2" ~ "Mortality:\nSHAP-informed",
    TRUE ~ long$model),
    levels = c("Readmission:\nNet risk",
               "Readmission:\nCum. incidence\nall predictors",
               "Readmission:\nCum. incidence\nSHAP-informed",
               "Mortality:\nAll predictors",
               "Mortality:\nSHAP-informed"))
  cols <- c("Readmission:\nNet risk" = "#9ECAE1",
            "Readmission:\nCum. incidence\nall predictors" = "#4292C6",
            "Readmission:\nCum. incidence\nSHAP-informed" = "#08519C",
            "Mortality:\nAll predictors" = "#B2182B",
            "Mortality:\nSHAP-informed" = "#E08214")
  shp  <- c("Readmission:\nNet risk" = 21,
            "Readmission:\nCum. incidence\nall predictors" = 23,
            "Readmission:\nCum. incidence\nSHAP-informed" = 25,
            "Mortality:\nAll predictors" = 24,
            "Mortality:\nSHAP-informed" = 22)
  pd <- position_dodge(width = dodge)
  theme_epi <- theme_classic(base_size = 13, base_family = tnr) +
    theme(legend.position = "bottom", legend.title = element_blank(), legend.key.width = unit(1.2, "lines"),
          strip.background = element_blank(), strip.text = element_text(face = "bold", size = 12),
          axis.title = element_text(face = "bold"), axis.text = element_text(color = "black"),
          panel.grid.major.y = element_line(color = "grey92", linewidth = 0.3),
          panel.spacing = unit(0.8, "lines"), plot.tag = element_text(face = "bold", size = 16, family = tnr))
  mk_panel <- function(m, ytitle, show_x = FALSE, show_strip = FALSE) {
    dd <- long[long$metric == m, ]
    g <- ggplot(dd, aes(horizon, value, color = model_lab, shape = model_lab, group = model_lab)) +
      geom_errorbar(aes(ymin = lower, ymax = upper), width = 2, linewidth = 0.5, alpha = 0.85, position = pd) +
      geom_line(linewidth = 0.7, alpha = 0.9, position = pd) +
      geom_point(size = 2.5, fill = "white", stroke = 0.9, position = pd) +
      facet_wrap(~ outcome, nrow = 1, scales = "free_y") +
      scale_color_manual(values = cols, drop = FALSE) + scale_shape_manual(values = shp, drop = FALSE) +
      scale_x_continuous(breaks = c(6, 12, 36, 60)) + expand_limits(y = 0) +
      labs(x = "Months since discharge", y = ytitle) + theme_epi
    if (!show_strip) g <- g + theme(strip.text = element_blank())
    if (!show_x) g <- g + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())
    g
  }
  ms <- names(metric_titles); nM <- length(ms)
  panels <- lapply(seq_along(ms), function(k)
    mk_panel(ms[k], metric_titles[ms[k]], show_x = (k == nM), show_strip = (k == 1)))
  patchwork::wrap_plots(panels, ncol = 1) + plot_layout(guides = "collect") +
    plot_annotation(tag_levels = "A") & theme(legend.position = "bottom")
}

if (exists("ici_boot")) {
  g_ici <- plot_ici_boot_epi(ici_boot, tnr); print(g_ici)
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot3_panel.tiff"), g_ici,
                  width = 17.8, height = 14, units = "cm", dpi = 600, compression = "lzw")
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot3_panel.png"), g_ici,
                  width = 17.8, height = 14, units = "cm", dpi = 600)
}


In [ ]:
#| label: holdout-ici-bootstrap-plot3-es
#| fig-width: 8
#| fig-height: 6.5

tnr <- "Times New Roman"

plot_ici_boot_epi <- function(ici_boot, tnr = "Times New Roman",
                              metric_titles = c(ici = "Índice de Calibración\nIntegrado (ICI)",
                                                ece = "Error de Calibracion\nEsperado (ECE)"),
                              dodge = 3) {
  d <- as.data.frame(ici_boot)
  mk <- function(m) data.frame(risk = d$risk, model = d$model, horizon = d$horizon, metric = m,
    value = d[[m]], lower = d[[paste0(m, "_lo")]], upper = d[[paste0(m, "_hi")]])
  long <- do.call(rbind, lapply(names(metric_titles), mk))
  long$outcome <- factor(ifelse(long$risk == "readmission", "Readmisi\u00f3n", "Mortalidad"),
                         levels = c("Readmisi\u00f3n", "Mortalidad"))
  long$model_lab <- factor(dplyr::case_when(
    long$model == "readmit::netrisk"  ~ "Readmisi\u00f3n:\nRiesgo neto",
    long$model == "readmit::bp1_cif"  ~ "Readmisi\u00f3n:\nIncid. acum.\nTodos pred.",
    long$model == "readmit::bp2_cif"  ~ "Readmisi\u00f3n:\nIncid. acum.\nSHAP-inform.",
    long$model == "death::best_perf1" ~ "Mortalidad:\nTodos los predictores",
    long$model == "death::best_perf2" ~ "Mortalidad:\nSHAP-inform.",
    TRUE ~ long$model),
    levels = c("Readmisi\u00f3n:\nRiesgo neto",
               "Readmisi\u00f3n:\nIncid. acum.\nTodos pred.",
               "Readmisi\u00f3n:\nIncid. acum.\nSHAP-inform.",
               "Mortalidad:\nTodos los predictores",
               "Mortalidad:\nSHAP-inform."))
  cols <- c("Readmisi\u00f3n:\nRiesgo neto" = "#9ECAE1",
            "Readmisi\u00f3n:\nIncid. acum.\nTodos pred." = "#4292C6",
            "Readmisi\u00f3n:\nIncid. acum.\nSHAP-inform." = "#08519C",
            "Mortalidad:\nTodos los predictores" = "#B2182B",
            "Mortalidad:\nSHAP-inform." = "#E08214")
  shp  <- c("Readmisi\u00f3n:\nRiesgo neto" = 21,
            "Readmisi\u00f3n:\nIncid. acum.\nTodos pred." = 23,
            "Readmisi\u00f3n:\nIncid. acum.\nSHAP-inform." = 25,
            "Mortalidad:\nTodos los predictores" = 24,
            "Mortalidad:\nSHAP-inform." = 22)
  pd <- position_dodge(width = dodge)
  theme_epi <- theme_classic(base_size = 13, base_family = tnr) +
    theme(legend.position = "bottom", legend.title = element_blank(), legend.key.width = unit(1.2, "lines"),
          strip.background = element_blank(), strip.text = element_text(face = "bold", size = 12),
          axis.title = element_text(face = "bold"), axis.text = element_text(color = "black"),
          panel.grid.major.y = element_line(color = "grey92", linewidth = 0.3),
          panel.spacing = unit(0.8, "lines"), plot.tag = element_text(face = "bold", size = 16, family = tnr))
  mk_panel <- function(m, ytitle, show_x = FALSE, show_strip = FALSE) {
    dd <- long[long$metric == m, ]
    g <- ggplot(dd, aes(horizon, value, color = model_lab, shape = model_lab, group = model_lab)) +
      geom_errorbar(aes(ymin = lower, ymax = upper), width = 2, linewidth = 0.5, alpha = 0.85, position = pd) +
      geom_line(linewidth = 0.7, alpha = 0.9, position = pd) +
      geom_point(size = 2.5, fill = "white", stroke = 0.9, position = pd) +
      facet_wrap(~ outcome, nrow = 1, scales = "free_y") +
      scale_color_manual(values = cols, drop = FALSE) + scale_shape_manual(values = shp, drop = FALSE) +
      scale_x_continuous(breaks = c(6, 12, 36, 60)) + expand_limits(y = 0) +
      labs(x = "Meses desde el egreso", y = ytitle) + theme_epi
    if (!show_strip) g <- g + theme(strip.text = element_blank())
    if (!show_x) g <- g + theme(axis.title.x = element_blank(), axis.text.x = element_blank(), axis.ticks.x = element_blank())
    g
  }
  ms <- names(metric_titles); nM <- length(ms)
  panels <- lapply(seq_along(ms), function(k)
    mk_panel(ms[k], metric_titles[ms[k]], show_x = (k == nM), show_strip = (k == 1)))
  patchwork::wrap_plots(panels, ncol = 1) + plot_layout(guides = "collect") +
    plot_annotation(tag_levels = "A") & theme(legend.position = "bottom")
}

if (exists("ici_boot")) {
  g_ici_es <- plot_ici_boot_epi(ici_boot, tnr); print(g_ici_es)
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot3_panel_es.tiff"), g_ici,
                  width = 17.8, height = 14, units = "cm", dpi = 600, compression = "lzw")
  ggplot2::ggsave(file.path(figs_out, "holdout_calibration_indices_boot3_panel_es.png"), g_ici,
                  width = 17.8, height = 14, units = "cm", dpi = 600)
}


In [ ]:
#| label: holdout-ici-bootstrap-plot4-epi
#| fig-width: 8
#| fig-height: 6.5
source(file.path(project_root, "cons/_alt_scripts/plot_calibration_indices_epi.R"), encoding = "UTF-8")

g_cal_idx <- plot_calibration_indices_epi(
  ici_boot,
  tnr          = "Times New Roman",
  panels       = c("ici", "eo"),
  figs_out     = figs_out,
  save         = TRUE,
  emit_caption = TRUE,
  eo_band      = NULL,
  eo_breaks    = c(0.8, 1, 1.25, 1.5),
  lang         = "en",
  model_labels = c(
    "readmit::netrisk" = "Readmission:\nNet risk", 
    "readmit::bp1_cif" = "Readmission:\nCum. incidence\nall predictors", 
    "readmit::bp2_cif" = "Readmission:\nCum. incidence\naSHAP-informed",
    "death::best_perf1" = "Mortality: all\npredictors",
    "death::best_perf2" = "Mortality: SHAP-\ninformed"
  )
)
print(g_cal_idx)
# caption prints below the chunk as a message; or place it explicitly:
message(attr(g_cal_idx, "caption"))

In [ ]:
#| label: holdout-ici-bootstrap-plot4-epi-es
#| fig-width: 8
#| fig-height: 6.5

g_cal_idx_es <- plot_calibration_indices_epi(
  ici_boot,
  tnr          = "Times New Roman",
  panels       = c("ici", "eo"),
  figs_out     = figs_out,
  save         = TRUE,
  emit_caption = TRUE,
  eo_band      = NULL,
  eo_breaks    = c(0.8, 1, 1.25, 1.5),
  lang         = "es",
  model_labels = c(
    "readmit::netrisk" = "Readmisi\u00f3n:\nRiesgo neto", 
    "readmit::bp1_cif" = "Readmisi\u00f3n:\nIncidencia acumulada\ntodos los predictores", 
    "readmit::bp2_cif" = "Readmisi\u00f3n:\nIncidencia acumulada\ninformada por SHAP",
    "death::best_perf1" = "Mortalidad: todos\nlos predictores",
    "death::best_perf2" = "Mortalidad: informado\npor SHAP"
  )
)
print(g_cal_idx_es)
# caption prints below the chunk as a message; or place it explicitly:
message(attr(g_cal_idx_es, "caption"))

## Threshold-based metrics

In [ ]:
#| label: holdout-threshold-bootstrap-pre
#| message: false

if (!exists("project_root")) project_root <- gsub("/cons$", "", here::here())
source(file.path(project_root, "cons/_alt_scripts/validate_holdout_metrics.R"))


In [ ]:
#| label: holdout-threshold-bootstrap-run
thr_boot_death <- threshold_bootstrap_holdout(
  list(best_perf1 = results_boot_val_bp1, best_perf2 = results_boot_val_bp2),
  horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile",
  freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE
)
thr_boot_readmit <- rbind(
  bootstrap_threshold_metrics_holdout(results_boot_val_bp1,     model_label = "readmit::netrisk", risks = "readmission", horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile", freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE),
  bootstrap_threshold_metrics_holdout(results_boot_val_bp1_cif, model_label = "readmit::bp1_cif",  risks = "readmission", horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile", freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE),
  bootstrap_threshold_metrics_holdout(results_boot_val_bp2_cif, model_label = "readmit::bp2_cif",  risks = "readmission", horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile", freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE)
)
thr_boot <- rbind(thr_boot_readmit[thr_boot_readmit$risk == "readmission", ], thr_boot_death[thr_boot_death$risk == "death", ])
utils::write.csv(thr_boot, file.path(out_dir, "pred23_holdout_threshold_bootstrap.csv"), row.names = FALSE)


In [ ]:
#| label: holdout-arm-comparison-table
source(file.path(project_root, "cons/_alt_scripts/holdout_arm_comparison.R"))
arms_readmit <- list(net_risk = results_boot_val_bp1, cif_bp1 = results_boot_val_bp1_cif, cif_bp2 = results_boot_val_bp2_cif)

cmp_prob <- .holdout_compare_predicted_prob(arms_readmit, val_list, times = c(6,12,36,60), B = 500L, seed = 2125L)
cmp_ibs  <- .holdout_compare_ibs(arms_readmit, eval_times = EVAL_TIMES, times = c(6,12,36,60),
              readmit_method = "aalen-johansen", B = 500L, seed = 2125L)
cmp_wolbers <- .holdout_compare_wolbers(results_boot_val_bp1_cif, results_boot_val_bp2_cif, val_list, horizons = c(6,12,36,60))

utils::write.csv(cmp_prob, file.path(out_dir, "pred23_holdout_arm_comparison_probability.csv"), row.names = FALSE)
utils::write.csv(cmp_ibs,  file.path(out_dir, "pred23_holdout_arm_comparison_ibs.csv"), row.names = FALSE)
utils::write.csv(cmp_wolbers, file.path(out_dir, "pred23_holdout_wolbers_sensitivity.csv"), row.names = FALSE)

In [ ]:
knitr::kable(cmp_prob[, c("arm_A","arm_B","horizon","diff","ij_lower","ij_upper","boot_lower","boot_upper","cv_A","cv_B")],
  digits = 4, caption = "Predicted-probability differences between readmission arms (IJ primary, bootstrap B=500 as check; CV secondary)")


In [ ]:
#| label: holdout-readmission-cif-ibs-absolute-bootstrap
#| message: true
#| warning: false
#| results: asis
# This chunk estimates absolute patient-bootstrap 95% confidence intervals for both competing-risk readmission CIF systems.
# BP1 combines the shared readmission model with the primary Full PH mortality model.
# BP2 combines the shared readmission model with the parsimonious SHAP mortality model.
# Both arms use the same bootstrap seed and therefore the same patient resamples.
# The fitted Cox models and individual predictions remain frozen throughout the bootstrap.
suppressPackageStartupMessages(library(survival))
if (!exists("project_root")) stop("project_root is missing. Run the notebook setup chunks first.", call. = FALSE)
if (!exists("out_dir")) out_dir <- file.path(project_root, "cons", "_out")
data_out <- file.path(project_root, "data", "20241015_out")
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(data_out, recursive = TRUE, showWarnings = FALSE)
required_objects <- c("results_boot_val_bp1_cif", "results_boot_val_bp2_cif")
missing_objects <- required_objects[!vapply(required_objects, exists, logical(1), inherits = TRUE)]
if (length(missing_objects) > 0L) stop("Missing CIF objects: ", paste(missing_objects, collapse = ", "), ". Run holdout-calibration-run first.", call. = FALSE)
source(file.path(project_root, "cons/_alt_scripts/ibs_window_bootstrap_holdout.R"))
IBS_CIF_B <- 1000L
IBS_CIF_SEED <- 2125L
IBS_CIF_GRID <- c(3, 6, 12, 36, 60)
IBS_CIF_HORIZONS <- c(6, 12, 36, 60)
IBS_CIF_ARMS <- list(`readmit::bp1_cif` = results_boot_val_bp1_cif, `readmit::bp2_cif` = results_boot_val_bp2_cif)
.run_absolute_cif_ibs <- function(results_object, model_label) {
message(sprintf("Running absolute CIF-based readmission IBS bootstrap for %s with B = %d and seed = %d.", model_label, IBS_CIF_B, IBS_CIF_SEED))
all_results <- ibs_window_bootstrap_core(results_object, B = IBS_CIF_B, seed = IBS_CIF_SEED, eval_times = IBS_CIF_GRID, readmit_method = "aalen-johansen", compute_null = FALSE, verbose = TRUE)
out <- all_results[all_results$risk == "readmission" & all_results$horizon %in% IBS_CIF_HORIZONS, c("risk", "horizon", "point", "mean", "q025", "q975")]
out <- out[match(IBS_CIF_HORIZONS, out$horizon), ]
stopifnot(nrow(out) == length(IBS_CIF_HORIZONS))
stopifnot(all(is.finite(as.matrix(out[c("point", "mean", "q025", "q975")]))))
out$bootstrap_bias <- out$mean - out$point
out$B <- IBS_CIF_B
out$seed <- IBS_CIF_SEED
out$model <- model_label
out$estimand <- "Competing-risk IBS based on the predicted CIF with death treated as a competing event"
out[c("model", "estimand", "horizon", "point", "q025", "q975", "mean", "bootstrap_bias", "B", "seed")]
}
ibs_cif_readmit_boot <- do.call(rbind, Map(.run_absolute_cif_ibs, IBS_CIF_ARMS, names(IBS_CIF_ARMS)))
rownames(ibs_cif_readmit_boot) <- NULL
stopifnot(nrow(ibs_cif_readmit_boot) == 2L * length(IBS_CIF_HORIZONS))
if (exists("cmp_ibs")) {
ref_bp1 <- cmp_ibs[cmp_ibs$arm_A == "net_risk" & cmp_ibs$arm_B == "cif_bp1", c("horizon", "mean_B")]
ref_bp2 <- cmp_ibs[cmp_ibs$arm_A == "net_risk" & cmp_ibs$arm_B == "cif_bp2", c("horizon", "mean_B")]
ref_bp1$model <- "readmit::bp1_cif"
ref_bp2$model <- "readmit::bp2_cif"
ref_points <- rbind(ref_bp1, ref_bp2)
reference_key <- paste(ref_points$model, ref_points$horizon)
result_key <- paste(ibs_cif_readmit_boot$model, ibs_cif_readmit_boot$horizon)
reference_index <- match(result_key, reference_key)
if (all(!is.na(reference_index))) {
max_abs_difference <- max(abs(ibs_cif_readmit_boot$point - ref_points$mean_B[reference_index]))
message(sprintf("Maximum point-estimate difference versus cmp_ibs across both CIF systems: %.3e.", max_abs_difference))
stopifnot(max_abs_difference < 1e-10)
}
}
ibs_cif_readmit_display <- data.frame(Model = ifelse(ibs_cif_readmit_boot$model == "readmit::bp1_cif", "Primary CIF, Full PH mortality", "Sensitivity CIF, SHAP mortality"), Horizon = ibs_cif_readmit_boot$horizon, `IBS (95% CI)` = sprintf("%.4f (%.4f to %.4f)", ibs_cif_readmit_boot$point, ibs_cif_readmit_boot$q025, ibs_cif_readmit_boot$q975), check.names = FALSE)
utils::write.csv(ibs_cif_readmit_boot, file.path(out_dir, "pred23_holdout_readmission_cif_ibs_absolute_bootstrap.csv"), row.names = FALSE)
saveRDS(ibs_cif_readmit_boot, file.path(data_out, "pred23_holdout_readmission_cif_ibs_absolute_bootstrap.rds"))
message("Absolute CIF-based readmission IBS bootstrap completed for BP1 and BP2.")
message("CSV saved to: ", file.path(out_dir, "pred23_holdout_readmission_cif_ibs_absolute_bootstrap.csv"))

In [ ]:
#| label: holdout-readmission-cif-ibs-absolute-bootstrap-print
print(knitr::kable(ibs_cif_readmit_display, format = "markdown", align = c("l", "c", "c"), caption = "Absolute competing-risk IBS for both readmission CIF systems, with patient-bootstrap 95% confidence intervals."))


In [ ]:
#| label: holdout-threshold-bootstrap-table

# Helper to render an HTML table with grouped rows.
# - body: data frame of values to display
# - outcome: grouping vector (e.g., "Readmission" / "Mortality")
# - caption: optional table caption
# - note: optional footnote below the table
threshold_browsable <- function(body, outcome, caption = NULL, note = NULL) {
  # Pull htmltools functions into the local namespace for brevity.
  tags <- htmltools::tags
  HTML <- htmltools::HTML
  tagList <- htmltools::tagList
  browsable <- htmltools::browsable
  # Generate a unique CSS class so multiple tables on the same page do not clash.
  cls <- paste0("thr-table-", as.integer(runif(1, 1e6, 9e6)))
  ncols <- ncol(body)
  # Build table rows as a list: one group header per outcome, then its rows.
  rows <- list()
  k <- 1L
  for (g in unique(outcome)) {
    # Rows belonging to this outcome group.
    idx <- which(outcome == g)
    # Add a bold spanning group header row.
    rows[[k]] <- tags$tr(
      class = "group-row",
      tags$td(colspan = ncols, g)
    )
    k <- k + 1L
    # Add each data row for the current group.
    for (i in idx) {
      rows[[k]] <- tags$tr(
        lapply(seq_len(ncols), function(j) {
          # Align column 1 left, columns 2-3 right, and the rest centered.
          al <- if (j == 1) "left" else if (j %in% c(2, 3)) "right" else "center"
          tags$td(style = paste0("text-align:", al, ";"), body[i, j, drop = TRUE])
        })
      )
      k <- k + 1L
    }
  }
  # Return a browsable HTML tagList with inline CSS and the table structure.
  browsable(tagList(
    tags$style(HTML(sprintf("
      .%s {
        border-collapse: collapse;
        margin: 0 auto;
        font-family: 'Times New Roman', serif;
        font-size: 12px;
        line-height: 1.25;
      }
      .%s caption {
        caption-side: top;
        font-weight: bold;
        margin-bottom: 8px;
      }
      .%s th, .%s td {
        border-bottom: 1px solid #dddddd;
        padding: 4px 8px;
        white-space: nowrap;
      }
      .%s th {
        border-bottom: 2px solid #777777;
        font-weight: bold;
      }
      .%s tbody tr:nth-child(even):not(.group-row) {
        background-color: #fafafa;
      }
      .%s tbody tr:not(.group-row):hover {
        background-color: #f3f6fb;
      }
      .%s .group-row td {
        background-color: #f3f3f3;
        font-weight: bold;
        text-align: left;
        border-top: 1px solid #bbbbbb;
      }
      .%s-note {
        max-width: 900px;
        margin: 8px auto 0 auto;
        font-family: 'Times New Roman', serif;
        font-size: 11px;
        line-height: 1.3;
      }
    ", cls, cls, cls, cls, cls, cls, cls, cls, cls))),
    tags$table(
      class = cls,
      if (!is.null(caption)) tags$caption(caption),
      tags$thead(tags$tr(lapply(names(body), tags$th))),
      tags$tbody(rows)
    ),
    if (!is.null(note)) tags$div(class = paste0(cls, "-note"), note)
  ))
}
# Prepare a tidy table of threshold metrics from a bootstrap thresholds object.
# - thr: threshold object, coerced to a data frame
# - digits: number of decimals for metrics and CI bounds
# - metrics: which metrics to include (Sens, Spec, PPV, NPV)
threshold_epi_table <- function(thr, digits = 2, metrics = c("Sens","Spec","PPV","NPV")) {
  # Convert to data frame and sort by outcome, model, horizon, and threshold.
  d <- as.data.frame(thr)
  d <- d[order(factor(ifelse(d$risk=="readmission","Readmission","Mortality"), c("Readmission","Mortality")),
               d$model, d$horizon, d$threshold), ]
  # Format a metric with its bootstrap percentile CI as "estimate (lo, hi)".
  # Missing values are shown as an em dash.
  f <- function(m) ifelse(is.na(d[[m]]), "\u2014",
        sprintf("%.*f (%.*f, %.*f)", digits, unname(d[[m]]),
                digits, unname(d[[paste0(m,"_lo")]]), digits, unname(d[[paste0(m,"_hi")]])))
  # Map abbreviated metric names to full labels.
  nm <- c(Sens="Sensitivity", Spec="Specificity", PPV="PPV", NPV="NPV")
  # Start the output table with core identifiers.
  out <- data.frame(
    Outcome = ifelse(d$risk=="readmission","Readmission","Mortality"),
    Model = dplyr::recode(d$model, "readmit::shared"="SHAP-informed (shared)",
      "death::best_perf1"="Full PH (best_perf1)", "death::best_perf2"="SHAP 13-var (best_perf2)"),
    Horizon = d$horizon, Threshold = sprintf("%g%%", 100*d$threshold),
    check.names = FALSE, stringsAsFactors = FALSE)
  # Append each requested metric column.
  for (m in metrics) out[[nm[m]]] <- f(m)
  # Return both the tidy table and the per-outcome group sizes for rendering.
  list(tab = out, grp = table(factor(out$Outcome, levels = unique(out$Outcome))))
}
# Build the bootstrap threshold table and remove repeated labels for cleaner display.
tt <- threshold_epi_table(thr_boot)
out <- tt$tab
# Composite keys used to blank out repeated model and horizon labels within outcomes.
km <- paste(out$Outcome, out$Model)
kh <- paste(km, out$Horizon)
# Blank out duplicated Model values within each outcome block.
out$Model   <- ifelse(duplicated(km), "", out$Model)
# Blank out duplicated Horizon values within each outcome-model block.
out$Horizon <- ifelse(duplicated(kh), "", as.character(out$Horizon))
# Drop the outcome column from the body; it is used as the group header instead.
body <- out[, -1]
names(body)[1:3] <- c("Model", "Horizon, mo", "Threshold")

In [ ]:
#| label: holdout-threshold-bootstrap-table2

threshold_epi_table <- function(thr, digits = 2, metrics = c("Sens","Spec","PPV","NPV")) {
  d <- as.data.frame(thr)
  d <- d[order(factor(ifelse(d$risk=="readmission","Readmission","Mortality"), c("Readmission","Mortality")),
               d$model, d$horizon, d$threshold), ]
  f <- function(m) ifelse(is.na(d[[m]]), "\u2014",
        sprintf("%.*f (%.*f, %.*f)", digits, unname(d[[m]]),
                digits, unname(d[[paste0(m,"_lo")]]), digits, unname(d[[paste0(m,"_hi")]])))
  nm <- c(Sens="Sensitivity", Spec="Specificity", PPV="PPV", NPV="NPV")
  out <- data.frame(
    Outcome = ifelse(d$risk=="readmission","Readmission","Mortality"),
    Model = dplyr::recode(d$model, "readmit::shared"="SHAP-informed (shared)",
      "death::best_perf1"="Full PH (best_perf1)", "death::best_perf2"="SHAP 13-var (best_perf2)"),
    Horizon = d$horizon, Threshold = sprintf("%g%%", 100*d$threshold),
    check.names = FALSE, stringsAsFactors = FALSE)
  for (m in metrics) out[[nm[m]]] <- f(m)
  list(tab = out, grp = table(factor(out$Outcome, levels = unique(out$Outcome))))
}

tt <- threshold_epi_table(thr_boot)
out <- tt$tab

km <- paste(out$Outcome, out$Model)
kh <- paste(km, out$Horizon)

out$Model   <- ifelse(duplicated(km), "", out$Model)
out$Horizon <- ifelse(duplicated(kh), "", as.character(out$Horizon))

body <- out[, -1]
names(body)[1:3] <- c("Model", "Horizon, mo", "Threshold")

threshold_browsable(
  body = body,
  outcome = out$Outcome,
  caption = "Held-out (20%) threshold-dependent metrics with bootstrap 95% CI.",
  note = "Predicted risk = 1-S(t) (cause-specific Cox). Readmission: competing-risk IPCW (Aalen-Johansen-aware; deaths before t weighted as controls); mortality: standard IPCW (Kaplan-Meier). 95% CI from bootstrap resampling of the held-out set. Readmission model shared by both."
)

## Save and session info

In [ ]:
#| label: session-info
#| echo: true
#| error: true
#| message: true
#| paged.print: true

message(paste0("R library: ", Sys.getenv("R_LIBS_USER")))
message(paste0("Date: ",withr::with_locale(new = c('LC_TIME' = 'C'), code =Sys.time())))
message(paste0("Editor context: ", getwd()))
cat("quarto version: "); quarto::quarto_version()
sesion_info <- devtools::session_info()

tabla_pkg <- dplyr::select(
  tibble::as_tibble(sesion_info$packages),
  package,
  loadedversion,
  source
)

tabla_pkg <- tibble::rowid_to_column(tabla_pkg, var = "row_number")

names(tabla_pkg) <- c("Row number", "Package", "Version", "Source")

htmltools::browsable(
  htmltools::tags$div(
    style = "
      max-height: 420px;
      overflow: auto;
      border: 1px solid #ddd;
      font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
      font-size: 70%;
      line-height: 0.75em;
      width: 100%;
    ",
    htmltools::tags$caption(
      style = "
        caption-side: top;
        text-align: left;
        display: block;
        padding: 6px 4px;
        font-size: 120%;
        line-height: 1.2em;
      ",
      htmltools::em("R packages")
    ),
    htmltools::tags$table(
      style = "
        border-collapse: collapse;
        width: max-content;
        min-width: 100%;
        white-space: nowrap;
      ",
      htmltools::tags$thead(
        htmltools::tags$tr(
          lapply(names(tabla_pkg), function(nm) {
            htmltools::tags$th(
              style = "
                position: sticky;
                top: 0;
                z-index: 2;
                background: #f3f3f3;
                border-bottom: 1px solid #ccc;
                padding: 4px 8px;
                text-align: left;
                white-space: nowrap;
              ",
              nm
            )
          })
        )
      ),
      htmltools::tags$tbody(
        lapply(seq_len(nrow(tabla_pkg)), function(i) {
          htmltools::tags$tr(
            lapply(tabla_pkg[i, ], function(x) {
              htmltools::tags$td(
                style = "
                  border-bottom: 1px solid #eee;
                  padding: 3px 8px;
                  white-space: nowrap;
                ",
                as.character(x)
              )
            })
          )
        })
      )
    )
  )
)

In [ ]:
#| label: holdout-save
holdout_validation <- list(
  created    = as.character(Sys.time()),
  seed       = 2125L,
  split_file = "cons/_out/comb_split_seed2125_test20_mar26.parquet"
)

# --- discrimination + prediction error (+ raw predictions = DCA/threshold source) ---
if (exists("results_boot_val_bp1") && exists("results_boot_val_bp2"))
  holdout_validation$results_boot_val <- list(best_perf1 = results_boot_val_bp1,
                                              best_perf2 = results_boot_val_bp2)
if (exists("cindex_ibs_global")) holdout_validation$cindex_ibs_global <- cindex_ibs_global
# --- calibration (curves + pooled ICI/ECE/E:O point estimates) ---
if (exists("cal_readmit"))   holdout_validation$cal_readmit <- cal_readmit
if (exists("cal_death_bp1") && exists("cal_death_bp2"))
  holdout_validation$cal_death <- list(best_perf1 = cal_death_bp1, best_perf2 = cal_death_bp2)
# --- DCA ---
if (exists("dca_models_full")) holdout_validation$dca_models_full <- dca_models_full
if (exists("dca_nb_bp1") && exists("dca_nb_bp2"))
  holdout_validation$dca_nb <- list(best_perf1 = dca_nb_bp1, best_perf2 = dca_nb_bp2)
# --- ipeval bootstrap (AUC / Brier / O:E + CIs) ---
if (exists("ipeval_res"))        holdout_validation$ipeval <- ipeval_res
# --- NRI / IDI reclassification ---
if (exists("nri_idi_holdout"))   holdout_validation$nri_idi <- nri_idi_holdout
if (exists("nri_idi_model_table")) holdout_validation$nri_idi_table <- nri_idi_model_table
# --- ICI / ECE bootstrap ---
if (exists("ici_boot"))          holdout_validation$ici_boot <- ici_boot
# --- threshold-dependent metrics bootstrap (Sens/Spec/PPV/NPV) ---
if (exists("thr_boot"))          holdout_validation$thr_boot <- thr_boot
# --- IBS per-window bootstrap (point + 95% CI, paired with Uno's C) ---
if (exists("ibs_window"))        holdout_validation$ibs_window <- ibs_window
# Bootstrap IBS
if (exists("ibs_cif_readmit_boot")) holdout_validation$ibs_cif_readmit_boot <- ibs_cif_readmit_boot
# DCA
if (exists("dca_boot_all")) holdout_validation$dca_boot_all <- dca_boot_all
# --- provenance ---
if (exists("models")) holdout_validation$models <-
  lapply(models, function(m) lapply(m, function(f) paste(deparse(f), collapse = " ")))
if (exists("hd")) holdout_validation$checks <- hd$checks

data_out <- file.path(project_root, "data", "20241015_out")
dir.create(data_out, recursive = TRUE, showWarnings = FALSE)

saveRDS(holdout_validation,
        file.path(data_out, paste0("pred23_holdout_validation_", format(Sys.Date(), "%Y_%m_%d"), ".rds")))

cat("Saved to", file.path(data_out, paste0("pred23_holdout_validation_", format(Sys.Date(), "%Y_%m_%d"), ".rds")), "\n")
cat("Top-level objects saved:\n"); print(names(holdout_validation))


In [ ]:
#| label: save-individual-plot-rds

plot_rds_dir <- file.path("data", "20241015_out", "pred23")
dir.create(plot_rds_dir, recursive = TRUE, showWarnings = FALSE)

plots_to_save <- list()

.add_existing <- function(save_name, object_name) {
  if (exists(object_name, inherits = TRUE)) {
    plots_to_save[[save_name]] <<- get(object_name, inherits = TRUE)
  } else {
    message("Skipping ", save_name, ": object not found: ", object_name)
  }
}

.add_expr <- function(save_name, expr) {
  p <- tryCatch(
    eval.parent(substitute(expr)),
    error = function(e) {
      message("Skipping ", save_name, ": ", conditionMessage(e))
      NULL
    }
  )
  if (!is.null(p)) plots_to_save[[save_name]] <<- p
  invisible(p)
}

# Plots that were printed but not assigned in earlier cells
if (exists("plot_metrics") && exists("results_boot_val_bp1")) {
  .add_expr("holdout_cindex_ibs_best_perf1", plot_metrics(results_boot_val_bp1$summary, NULL))
}
if (exists("plot_metrics") && exists("results_boot_val_bp2")) {
  .add_expr("holdout_cindex_ibs_best_perf2", plot_metrics(results_boot_val_bp2$summary, NULL))
}

if (exists("cal_curve_plot") && exists("cal_readmit")) {
  .add_expr("holdout_calibration_readmit_basic", cal_curve_plot(cal_readmit, NULL, "#2166AC"))
}
if (exists("cal_curve_plot") && exists("cal_death_bp1")) {
  .add_expr("holdout_calibration_death_best_perf1_basic", cal_curve_plot(cal_death_bp1, NULL, "#B2182B"))
}
if (exists("cal_curve_plot") && exists("cal_death_bp2")) {
  .add_expr("holdout_calibration_death_best_perf2_basic", cal_curve_plot(cal_death_bp2, NULL, "#B2182B"))
}

if (exists(".make_cal_panel") && exists("cal_readmit")) {
  .add_expr(
    "holdout_calibration_readmit_epi_panel",
    .make_cal_panel(cal_readmit, color = "#2166AC", x_lim = c(0, 0.4), x_by = 0.1)
  )
}

# Plot objects already assigned by the notebook
.add_existing("holdout_calibration_readmit_indices", "fig_r")
.add_existing("holdout_calibration_death_best_perf1_indices", "fig_d1")
.add_existing("holdout_calibration_death_best_perf2_indices", "fig_d2")

if (all(vapply(c("fig_r", "fig_d1", "fig_d2"), exists, logical(1), inherits = TRUE))) {
  fig_cal_indices_all <- .add_expr(
    "holdout_calibration_indices_three_panel",
    patchwork::wrap_plots(fig_r, fig_d1, fig_d2, ncol = 1) +
      patchwork::plot_annotation(
        tag_levels = "A",
        theme = ggplot2::theme(
          plot.tag = ggplot2::element_text(
            face = "bold",
            family = if (exists("tnr", inherits = TRUE)) get("tnr", inherits = TRUE) else ""
          )
        )
      )
  )
}

# Raw DCA panels (printed but not assigned)
if (exists("make_dca_panel_figure") && exists("dca_models_full")) {
  .add_expr("holdout_dca_readmit_raw",
    make_dca_panel_figure(dca_models_full$best_perf1$summary,
                          dca_models_full$best_perf2$summary,
                          outcome = "readmission", horizons = c(12, 36, 60)))
  .add_expr("holdout_dca_death_raw",
    make_dca_panel_figure(dca_models_full$best_perf1$summary,
                          dca_models_full$best_perf2$summary,
                          outcome = "death", horizons = c(12, 36, 60)))
}

# Standardized DCA panels
.add_existing("holdout_calibration_curves_three_panel", "g_cal_curves")
.add_existing("holdout_dca_abc", "p_dca_abc")
.add_existing("holdout_dca_readmit_std", "p_readmit_panel_hold")
.add_existing("holdout_dca_death_std",   "p_death_panel_hold")

# Epi calibration indices panel
.add_existing("holdout_calibration_indices_epi", "g_cal_idx")

# inside cell 59, after print(g_ici)
saveRDS(g_ici, file.path(plot_rds_dir, "holdout_ici_bootstrap_boot2_panel.rds"))

.add_existing("holdout_mortality_calibration_best_perf1_epi", "death_A")
.add_existing("holdout_mortality_calibration_best_perf2_epi", "death_B")
.add_existing("holdout_mortality_calibration_AB", "final_death")
.add_existing("holdout_ipeval_bootstrap", "g_ip")
.add_existing("holdout_ici_bootstrap", "g_ici")

.safe_name <- function(x) gsub("[^A-Za-z0-9_.-]+", "_", x)

saved_plot_rds <- vapply(names(plots_to_save), function(nm) {
  f <- file.path(plot_rds_dir, paste0(.safe_name(nm), ".rds"))
  saveRDS(plots_to_save[[nm]], f)
  f
}, character(1))

cat("Saved individual plot RDS files to:\n", normalizePath(plot_rds_dir, winslash = "/"), "\n")
print(saved_plot_rds)